<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="https://www.uoc.edu/content/dam/news/images/noticies/2016/202-nova-marca-uoc.jpg" align="left" width="45%">
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">M2.878 · Trabajo de Fin de Máster · AED</p>
<p style="margin: 0; text-align:right;">2025-2 · Máster universitario en Ciencia de datos</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Marcos Rodríguez Soler</p>
</div>
</div>
<div style="width:100%;">&nbsp;</div>

# Análisis Exploratorio de Datos

En este _notebook_ se lleva a cabo el **Análisis Exploratorio de los Datos** que se estructura en los siguientes subapartados.

<ol style="list-style: none; padding-left: 0;">
  <li>1. <a href="#ej1">Integración de los datos</a> <br>
    &nbsp;&nbsp;1.1. <a href="#ej1.1">Importación de los datos</a> <br>
    &nbsp;&nbsp;1.2. <a href="#ej1.2">Unión de los datos</a>
  </li>

  <li>2. <a href="#ej2">Exploración del conjunto de datos</a> <br>
    &nbsp;&nbsp;2.1. <a href="#ej2.1">Inspección general del dataset</a> <br>
    &nbsp;&nbsp;2.2. <a href="#ej2.2">Análisis de calidad de los datos</a> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;2.2.1. <a href="#ej2.2.1">Valores nulos</a> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;2.2.2. <a href="#ej2.2.2">Análisis de rangos</a> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;2.2.3. <a href="#ej2.2.3">Inconsistencias</a> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;2.2.4. <a href="#ej2.2.4">Roturas de <i>stock</i></a>
  </li>

  <li>3. <a href="#ej3">Análisis de las variables</a> <br>
    &nbsp;&nbsp;3.1. <a href="#ej3.1">Análisis de las variables auxiliares</a> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;3.1.1. <a href="#ej3.1.1"><i>producto</i></a> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;3.1.2. <a href="#ej3.1.2"><i>idSecuencia</i></a> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;3.1.3. <a href="#ej3.1.3"><i>isPromo</i></a> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;3.1.4. <a href="#ej3.1.4"><i>bolOpen</i></a> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;3.1.5. <a href="#ej3.1.5"><i>bolHoliday</i></a> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;3.1.6. <a href="#ej3.1.6"><i>diasEntrePedidos</i></a> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;3.1.7. <a href="#ej3.1.7"><i>diasLeadtime</i></a> <br>
    &nbsp;&nbsp;&nbsp;&nbsp;3.1.8. <a href="#ej3.1.8"><i>eurPrecioMedio</i></a> <br>
    &nbsp;&nbsp;3.2. <a href="#ej3.2">Análisis de la variable objetivo</a> <br>
    &nbsp;&nbsp;3.3. <a href="#ej3.3">Análisis conjunto</a>
  </li>

  <li>4. <a href="#ej4"><i>Clustering</i></a> <br>
  </li>
</ol>

In [ ]:
import holidays

import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib
from matplotlib.lines import Line2D
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize, Colormap, ListedColormap

from statsmodels.tsa.stattools import adfuller

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from typing import Any, Set, Dict, List, Tuple
from collections import defaultdict

%matplotlib inline

<br><br><a id='ej1'></a>
# 1. Integración de los datos

En este apartado se integran los datos provenientes de los siguientes tres archivos _xlsx_:

1. _DatosCicloAprovisionamiento.xlsx_ &#8594; Información acerca del ciclo de reposición de cada producto. Contiene los campos _producto_, referente al identificador de cada producto, _diasEntrePedidos_, días del ciclo de revisión de cada producto, y _diasLeadTime_, que es el tiempo de entrega correspondiente a cada ítem. 
2. _DatosPrecioMedio.xlsx_ &#8594; Datos referentes a los precios de cada ítem. Contiene los atributos _producto_, siendo el identificador de cada producto, y _eurPrecioMedio_, que es el promedio del precio de cada ítem.
3. _Datos.xlsx_ &#8594; Este es el fichero principal de datos, y está compuesto por cuatro hojas distintas:
   <ul>
       <li>Hoja <i>Venta</i> &#8594; Información acerca de la demanda de cada producto. Incluye las variables <i>producto</i>, que identifica a cada ítem de forma únivoca, <i>idSecuencia</i>, que se corresponde con una fecha en formato <i>YYYYmmdd</i>, y <i>udsVenta</i>, que son el número de ventas de un producto concreto en una fecha determinada.</li>
       <li>Hoja <i>Calendario</i> &#8594; Información relevante de las fechas del calendario. Contiene los atributos <i>idSecuencia</i>, que es referente a una fecha, <i>bolOpen</i>, que indica si la tienda se encontraba abierta en una fecha concreta, y <i>bolHoliday</i>, que denota si la fecha se corresponde a una festividad.</li>
       <li>Hoja <i>Promociones</i> &#8594; Datos acerca de los períodos promocionales de cada producto. Incluye los campos <i>producto</i>, referente al identificador único de cada ítem, <i>idSecuenciaIni</i>, que es la fecha inicial de la promoción, y <i>idSecuenciaFin</i>, que se corresponde con la fecha del fin del período promocional.</li>
       <li>Hoja <i>Stock</i> &#8594; Información del inventario disponible de cada producto para un conjunto de fechas. Contiene las variables <i>producto</i>, para identificar cada elemento a la venta, <i>idSecuencia</i>, que se corresponde con una fecha, y <i>udsStock</i>, referente a las unidades disponibles de un producto en una fecha concreta.</li>
   </ul>

El objetivo de este apartado consiste en la integración de todos estos archivos y hojas en un solo conjunto de datos, donde cada fila debe representar la demanda de un producto concreto en una fecha concreta, agregando información adicional tanto del ítem como de la fecha. Consecuentemente, este proceso de integración se debe hacer respecto a la hoja _Venta_ del archivo _Datos.xslx_, donde se deben ir uniendo los otros ficheros u hojas mediante operaciones del tipo _**LEFT JOIN**_ de SQL para siempre tener constancia de las ventas de cada producto.

<a id='ej1.1'></a>
## 1.1. Importación de los datos

El primer paso para integrar todos los datos en un solo conjunto consiste en la importación individual de los archivos y hojas anteriormente mencionados. Para ello, se utiliza la función _**importar_datos()**_ que se define a continuación. Este procedimiento se encarga de cargar de forma automática los tres archvos _xlsx_ con los que se va a trabajar, es decir, _DatosCicloAprovisionamiento.xlsx_, _DatosPrecioMedio.xlsx_, y todas las hojas de _Datos.xlsx_. Asimismo, cabe destacar que para el correcto funcionamiento de la función, los datos deben almacenarse en una carpeta distinta a la que contiene este _notebook_ pero que esté en la misma jerarquía.

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── DIRECTORIO_CON_LOS_DATOS <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└── DIRECTORIO_ACTUAL

In [ ]:
def importar_datos(ruta_archivo: str, nombre_hoja=0, primeras_filas=5) -> pd.DataFrame:
    """Importa una hoja de un archivo Excel concreto y muestra algunos datos básicos para garantizar que la
    importación se ha llevado a cabo existosamente.

    Argumentos:
        ruta_archivo: str -> Ruta del archivo Excel que se desea importar.
        nombre_hoja -> Nombre de la hoja del archivo que se desea importar. Por defecto es 0, lo cual implica
        que el fichero tan solo contiene una sola hoja.
        primeras_filas -> Número del total de primeras filas que se desea imprimir

    Devuelve:
        pd.DataFrame -> DataFrame de Pandas con los datos importados.
    """
    # Nombre del archivo
    nombre: str = ruta_archivo.split("/")[2]
    nombre_completo: str = (
        f"El archivo \"{nombre}\"" 
        if nombre_hoja == 0 
        else f"La hoja \"{nombre_hoja}\" del archivo \"{nombre}\""
    )
    
    # Importación
    dataset: pd.DataFrame = pd.read_excel(ruta_archivo, sheet_name=nombre_hoja)
    
    # Dimensiones
    dim1, dim2 = dataset.shape
    print(f"{nombre_completo} contiene {dim1} instancias y {dim2} atributos")
    
    # Variables
    campos: List[str] = dataset.columns
    print(f"{nombre_completo} incluye las variables \"{', '.join(campos)}\"", "\n")
    
    # Tipos de datos de cada variable y porcentaje de valores nulos
    tipos_datos: List[str] = [dataset[campo].dtype for campo in campos]
    nulos: pd.Series = dataset.isna().mean() * 100
    for campo, nulo, tipo_dato in zip(campos, nulos, tipos_datos):
        print(f"La variable \"{campo}\" contiene un {nulo}% de valores nulos y es de tipo \"{tipo_dato}\"")
    
    # Primeros registros
    print("\n")
    print(f"Se muestran los primeros {primeras_filas} registros", "\n", dataset.head(primeras_filas))

    return dataset

<br><br>Así pues, se procede con la carga de los datos del archivo _DatosCicloAprovisionamiento.xlsx_.

In [ ]:
# Archivo DatosCicloAprovisionamiento.xlsx
ruta_datos_ciclo: str = "../DATOS/DatosCicloAprovisionamiento.xlsx"
datos_ciclo: pd.DataFrame = importar_datos(ruta_datos_ciclo)

<br><br>Se prosigue con el archivo _DatosPrecioMedio.xlsx_.

In [ ]:
# Archivo DatosPrecioMedio.xlsx
ruta_datos_precio: str = "../DATOS/DatosPrecioMedio.xlsx"
datos_precio: pd.DataFrame = importar_datos(ruta_datos_precio)

<br><br>Se continua con la hoja _Venta_ del archivo _Datos.xlsx_.

In [ ]:
# Hoja Venta del archivo Datos.xlsx
ruta_datos: str = "../DATOS/Datos.xlsx"
nombre_hoja: str = "Venta"
datos_ventas: pd.DataFrame = importar_datos(ruta_datos, nombre_hoja=nombre_hoja)

<br><br>Se sigue la importación de datos con la hoja _Calendario_ del archivo _Datos.xlsx_.

In [ ]:
# Hoja Calendario del archivo Datos.xlsx
nombre_hoja: str = "Calendario"
datos_calendario: pd.DataFrame = importar_datos(ruta_datos, nombre_hoja=nombre_hoja)

<br><br>Se prosigue con la hoja _Promociones_ del archivo _Datos.xlsx_.

In [ ]:
# Hoja Calendario del archivo Datos.xlsx
nombre_hoja: str = "Promociones"
datos_promociones: pd.DataFrame = importar_datos(ruta_datos, nombre_hoja=nombre_hoja)

<br><br>Por último, se terminan las importaciones con la hoja _Stock_ del archivo _Datos.xlsx_.

In [ ]:
# Hoja Stock del archivo Datos.xlsx
nombre_hoja: str = "Stock"
datos_stock: pd.DataFrame = importar_datos(ruta_datos, nombre_hoja=nombre_hoja)

Las importaciones han tenido lugar de forma existosa, ya que tanto las dimensiones de los conjuntos, como los atributos y los tipos de cada variable se corresponden con los contenidos de las mismas hojas y archivos _xlsx_. Asimismo, no se han encontrado valores nulos en ningún conjunto de datos de manera que, por el momento, facilita el proceso de integración. Adicionalmente, cabe destacar el hecho que las variables referentes a fechas se han codificado a tipo _int64_ a pesar de que representan una fecha. Esto ahora mismo no supone un problema ya que estos campos se usarán como clave para cruzar los distintos conjuntos de datos extraídos.

<a id='ej1.2'></a>
## 1.2. Unión de los datos

Una vez se han extraído los contenidos de las hojas y archivos _xlsx_, se procede con la unión de los distintos conjuntos empleando como referencia la hoja _Venta_ del archivo _Datos.xlsx_ ya que, como se mencionó anteriormente, esta hoja es la que contiene la información más importante para predecir la demanda futura, siendo la propia demanda que tuvo lugar en el pasado. Sin embargo, antes de fusionar los conjuntos se debe tener presente que la unión con la hoja _Promociones_ no es trivial, dado que no existe un atributo que sirva de punto de unión entre ambas tablas. Así pues, la unión se debe hacer en este caso mediante una máscara _booleana_ que se pasa al _dataset_ de ventas, donde se filtran aquellas filas cuya ID del producto se corresponda con un ítem sujeto a promoción, basándose en el período promocional que comprende el intervalo que va desde _idSecuenciaIni_ hasta _idSecuenciaFin_, donde ambos campos pertenece a la hoja _Promociones_. Esto da lugar a la variable _isPromo_ que denota si un producto concreto en una fecha determinada estaba sujeto a promoción.

In [ ]:
# Adición de la variable isPromo
for _, fila in datos_promociones.iterrows():
    id_producto: int = fila["producto"]
    fecha_inicial: int = fila["idSecuenciaIni"]
    fecha_final: int = fila["idSecuenciaFin"]

    condicion: pd.Series = (
        (datos_ventas["producto"] == id_producto) &
        (datos_ventas["idSecuencia"] >= fecha_inicial) &
        (datos_ventas["idSecuencia"] <= fecha_final)
    )

    datos_ventas.loc[condicion, "isPromo"] = 1

datos_ventas["isPromo"] = datos_ventas["isPromo"].fillna(0).astype(int)

<br><br>Tras haber plasmado la información de la hoja _Promociones_ en el conjunto de datos _datos_ventas_, se termina con la unión con las otras hojas y archivos.

In [ ]:
# Unión con la hoja Calendario
dataset_temporal_1: pd.DataFrame = datos_ventas.merge(
    datos_calendario,
    how="left",
    left_on="idSecuencia",
    right_on="idSecuencia"
)

# Unión con la hoja Stock
dataset_temporal_2: pd.DataFrame = dataset_temporal_1.merge(
    datos_stock,
    how="left",
    left_on=["producto", "idSecuencia"],
    right_on=["producto", "idSecuencia"]
)

# Unión con el archivo DatosCicloAprovisionamiento.xlsx
dataset_temporal_3: pd.DataFrame = dataset_temporal_2.merge(
    datos_ciclo,
    how="left",
    left_on="producto",
    right_on="producto"
)

# Unión con el archivo DatosPrecioMedio.xlsx
dataset: pd.DataFrame = dataset_temporal_3.merge(
    datos_precio,
    how="left",
    left_on="producto",
    right_on="producto"
)

<br><br><a id='ej2'></a>
# 2. Exploración de los datos

Una vez se han integrado los datos en un conjunto global ya se dispone de suficiente información para iniciar formalmente la etapa del análisis exploratorio.

<a id='ej2.1'></a>
## 2.1. Inspección general del _dataset_

Esta sección sirve para adquirir una noción fundamental de la estructura del _dataset_ global tras la unión de las distintas hoja y archivos del apartado anterior. El contexto de los datos se sitúa en una empresa de productos de distribución para el automóvil, donde los datos de los productos se han anonimizado por motivos de confidencialidad.

In [ ]:
# Información básica del conjunto de datos global
dataset.info()

In [ ]:
# Primeras 5 filas del dataset global
dataset.head()

In [ ]:
# Se comprueba que no existen registros duplicados
if not all(dataset[["producto", "idSecuencia"]].duplicated()):
    print("El conjunto de datos no contiene ninguna muestra duplicada")

El conjunto de datos ahora cuenta con 10 variables distintas. Asimismo, el número total de registros se sigue manteniendo constante respecto al _dataset_ de nombre _datos_ventas_, dado que las uniones se han hecho respecto a esta misma tabla. Por otro lado, cabe destacar que ahora sí aparecen valores faltantes en los datos, ya que es lo que ocurre cuando se unen dos _DataFrames_ con un _**LEFT JOIN**_ y no se encuentran coincidencias entre la tabla de la izquierda y la de la derecha, resultando en valores nulos en los atributos del conjunto de la derecha. En este contexto, se ofrece una descripción breve de los campos que conforman el nuevo conjunto de datos global. Asimismo, es importante destacar el hecho que cada fila representa la demanda de un producto en una fecha concreta, y que el _dataset_ se estructura a nivel producto-día.
    <ul>
        <li><i>producto</i> &#8594; Identificador único de cada ítem del negocio</li>
        <li><i>idSecuencia</i> &#8594; Fecha en formato entero de la que se dispone de la demanda de un producto concreto</li>
        <li><i>udsVenta</i> &#8594; Demanda de un producto a lo largo de una fecha concreta</li>
        <li><i>isPromo</i> &#8594; Indica si el ítem a la venta formaba parte de una promoción en una fecha concreta</li>
        <li><i>bolOpen</i> &#8594; Denota si la tienda del negocio estaba abierta en una fecha concreta</li>
        <li><i>bolHoliday</i> &#8594; Indica si una fecha concreta formaba parte de un período vacacional</li>
        <li><i>udsStock</i> &#8594; Número de unidades en el almacén de un producto concreto en una fecha determinada</li>
        <li><i>diasEntrePedidos</i> &#8594; Intervalo de tiempo en días que debe pasar para realizar pedidos de un producto concreto</li>
        <li><i>diasLeadtime</i> &#8594; Tiempo de entrega en días de un ítem determinado</li>
        <li><i>eurPrecioMedio</i> &#8594; Promedio del precio de un producto</li>
    </ul>
Es muy importante comentar el hecho que la variable _udsStock_ no se puede emplear en los modelos predictivos, ya que contiene información que no es conocida a futuro y que, por lo tanto, no puede usarse para realizar predicciones. No obstante, aporta información útil para comprender algunos fenómenos que se dan en el _dataset_ como se mostrará más adelante.

Por último, se realiza un breve análisis estadístico descriptivo de las variables numéricas del conjunto de datos.

In [ ]:
# Resumen estadístico de las variables numéricas del dataset
dataset[["udsVenta", "udsStock", "eurPrecioMedio", "diasLeadtime", "diasEntrePedidos"]].describe().round(2).T

A partir de los estadísticos descriptivos de las variables numéricas se pueden extraer varias conclusiones relevantes. En primer lugar, la variable _udsVenta_ presenta una distribución altamente asimétrica, tal y como se observa por la diferencia entre la media y la mediana, así como por la presencia de valores máximos muy elevados de hasta 397 unidades. Esto indica que la demanda se caracteriza por una gran cantidad de días con ventas nulas o muy bajas, como se observó en el apartado [Roturas de _stock_](#ej2.2.4), combinados con picos puntuales de alta demanda. En cuanto a la variable _eurPrecioMedio_, se observa una elevada dispersión, lo que refleja una considerable heterogeneidad en el catálogo de productos, con artículos de bajo coste y otros significativamente más caros. Por otro lado, las variables _diasLeadtime_ y _diasEntrePedidos_ presentan una variabilidad más contenida, lo que sugiere una mayor estabilidad en los procesos logísticos. En particular, _diasEntrePedidos_ muestra una concentración clara entorno a 14 días, indicando un patrón de reposición bastante regular para la mayoría de productos.

Por otro lado, se crea la variable _fecha_ que representa el mismo concepto que _idSecuencia_ pero está codificada a un tipo de dato _datetime_ de _Pandas_, lo cual resulta útil para generar gráficos para comprender los horizontes de la demanda que cubre el _dataset_.

In [ ]:
# Campo fecha
dataset["fecha"] = pd.to_datetime(dataset["idSecuencia"], format="%Y%m%d")

<a id='ej2.2.1'></a>
### 2.2.1. Valores nulos

Uno de los aspectos más importantes a tener en cuenta durante la exploración de los datos es la detección de anomalías, en este caso referiéndose a la presencia de valores nulos. A pesar de que los conjuntos individuales que se trataron en el apartado [Importación de los datos](#ej1.1) no presentaban valores faltantes, la unión entre ellos con la operación _**LEFT JOIN**_ sí que da origen a estas valores si no se encuentran correspondencias entre los juegos de datos que se asocian. En este sentido, es importante tener presente el porcentaje de nulos por variable, su valor absoluto por atributo, y los patrones de valores ausentes por fila.

In [ ]:
print("Proporción de nulos por variable\n", dataset.isna().mean() * 100, "\n")
print("Total de nulos por variable\n", dataset.isna().sum())

In [ ]:
# Gráfico de barras con los porcentajes de valores nulos por variable
plt.bar(dataset.isna().mean().index.tolist(), dataset.isna().mean() * 100, color="blue", alpha=0.6, edgecolor="black", linewidth=1)

plt.title("Porcentaje de valores nulos por variable")
plt.xlabel("Variables")
plt.tick_params(axis="x", rotation=45)
plt.ylabel("Porcentaje de valores nulos")
plt.yticks([i/10 for i in range(0, 4, 1)])

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Patrones de nulos por fila
patrones: Dict[str, int] = defaultdict(int)
for _, fila in dataset.iterrows():

    estado_apertura: str = "1" if np.isnan(fila["bolOpen"]) else "0"
    estado_vacaciones: str = "1" if np.isnan(fila["bolHoliday"]) else "0"
    estado_inventario: str = "1" if np.isnan(fila["udsStock"]) else "0"
    patron_nulos: str = estado_apertura + estado_vacaciones + estado_inventario

    patrones[patron_nulos] += 1

print("Patrones de valores nulos por fila:\n")
for clave, valor in patrones.items():
    print(f"{clave} filas con el patrón {valor}")

In [ ]:
plt.bar(patrones.keys(), patrones.values(), color="blue", alpha=0.6, edgecolor="black", linewidth=1)

plt.title("Patrones de nulos por fila")
plt.xlabel("Patrones (bolOpen - bolHoliday - udsStock)")
plt.ylabel("Frecuencia")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

Las columnas _bolOpen_, _bolHoliday_, y _udsStock_ son las únicas que contienen estos valores, además en la misma proporción. Asimismo, existe un único patrón de valores nulos por fila en todo el conjunto de datos, que es cuando las tres variables mencionadas anteriormente contienen valores ausentes simultáneamente en la misma fila. Esto apunta a que la aparición de estos valores está fuertemente relacionada con la estructura de las dos tablas que se han unido en el apartado [Unión de los datos](#ej1.2), siendo estas la hoja _Ventas_ del archivo _Datos.xlsx_ y la hoja _Calendario_ del mismo archivo. Dado que ambos conjuntos de datos se han unido a través del campo _idSecuencia_, este hecho indica que hay fechas de la demanda histórica de las que no se dispone de la información de _bolOpen_ y _bolHoliday_. De forma parecida, la unión de la hoja _Ventas_ con la hoja _Stock_ del propio fichero _Datos.xlsx_ se realizó empleando los atributos _producto_ e _idSecuencia_ como punto de unión, de manera que también existen fechas de las que no se dispone del inventario en aquel momento.

In [ ]:
# Fechas asociadas a las filas con nulos
na_dataset: pd.DataFrame = dataset.loc[dataset.isna().any(axis=1)]
fechas_con_nulos: pd.Series = na_dataset["idSecuencia"].unique()

print("Las fechas asociadas a las filas con valores nulos son:\n")
for fecha in fechas_con_nulos:
    print(fecha)

In [ ]:
# Nulos por producto
nulos_producto: pd.DataFrame = dataset.groupby("producto", as_index=False).agg(
    total_nulos=(
        "bolOpen",
        lambda g: g.isna().sum()
    )
)

ax: matplotlib.axes.Axes = nulos_producto.plot(
    kind="scatter",
    x="producto",
    y="total_nulos",
    color="blue",
    alpha=0.6
)

ax.set_title("Valores nulos por producto")
ax.set_xlabel("producto")
ax.set_ylabel("Valores nulos")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

El hecho de que el día 2024-03-22 sea la única fecha de la que no se dispone de _bolOpen_, _bolHoliday_ y _udsStock_ denota que existe un problema con la fuente de datos, donde quizás se olvidó añadir los registros de las variables destacadas para esta fecha concreta.

Así pues, tras entender tanto los valores nulos del conjunto de datos como su origen, se decide eliminar todos los registros del _dataset_ con este tipo de datos. Se considera que el impacto negativo que esto tiene es mínimo, ya que tan solo se está eliminando una fecha por producto y los intervalos de la demanda histórica abarcan mucho más tiempo. Asimismo, el origen de estos datos no es aleatorio y está relacionado con la propia fuente de los estos. Por otro lado, estos valores aparecen en variables importantes que se utilizarán más adelante en el proceso de modelado y, además, se considera que un proceso de imputación sería excesivamente complejo para el impacto despreciable que tendría, donde posiblemente se terminara añadiendo una pequeña fuente de sesgo en los datos.

In [ ]:
# Eliminación de filas con nulos
dataset.dropna(inplace=True)

<a id='ej2.2.2'></a>
### 2.2.2 Análisis de rangos

Otro indicador de la calidad de los datos son los rangos numéricos que estos abarcan. Estos intervalos deben comprender un conjunto de valores que sea coherente con la naturaleza de cada variable ya que, por ejemplo, resularía ilógico encontrar un valor negativo de la variable _eurPrecioMedio_. Para analizar esta característica de los datos, resulta especialmente útil la visualización tanto de los histogramas como de los _boxplots_ de cada atributo.

In [ ]:
# Histogramas de las variables del dataset
vars: List[str] = [
    "producto", "diasLeadtime", "diasEntrePedidos", "udsVenta", "udsStock", "eurPrecioMedio",
    "isPromo", "bolHoliday", "bolOpen", "", "fecha"
]

fig, axs = plt.subplots(nrows=4, ncols=3, figsize=(18, 20))
axs = axs.flatten()

for i, (ax, var) in enumerate(zip(axs, vars)):
    if i == 9:
        continue
    
    if var in ["producto", "diasEntrePedidos", "diasLeadtime"]:
        dataset[var].hist(ax=ax, bins=30, alpha=0.6, edgecolor="black", linewidth=1, color="blue")
        frecuencia: str = "Frecuencia"
        ax.axvline(dataset[var].mean(), color="blue", linestyle="--", linewidth=2)
    elif var == "fecha":
        dataset[var].hist(ax=ax, alpha=0.6, edgecolor="black", linewidth=1, color="blue")
        frecuencia: str = "Frecuencia"
        ax.tick_params("x", rotation=45)
    elif var in ["isPromo", "bolOpen", "bolHoliday"]:
        dataset[var].hist(ax=ax, bins=2, alpha=0.6, edgecolor="black", linewidth=1, color="blue")
        frecuencia: str = "Frecuencia"
        ax.set_xticks([0, 1])
    else:
        dataset[var].hist(ax=ax, log=True, bins=30, alpha=0.6, edgecolor="black", linewidth=1, color="blue")
        frecuencia: str = "log(Frecuencia)"
        ax.axvline(dataset[var].mean(), color="blue", linestyle="--", linewidth=2)

    ax.set_title(f"{var}")
    ax.set_xlabel(f"Valores de {var}")
    ax.set_ylabel(frecuencia)

    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots de las variables numéricas
num_vars: List[str] = ["diasLeadtime", "diasEntrePedidos", "udsVenta", "udsStock", "eurPrecioMedio"]

fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(14, 8))
axs = axs.flatten()

for ax, num_var in zip(axs, num_vars):
    dataset[num_var].plot(
        kind="box",
        ax=ax,
        patch_artist=True,
        boxprops=dict(facecolor="blue", alpha=0.6, color="black", linewidth=1),
        medianprops=dict(color="black", linewidth=2),
        whiskerprops=dict(color="black", linewidth=1),
        capprops=dict(color="black", linewidth=1)
    )

    ax.axhline(dataset[num_var].mean(), color="blue", linestyle="--", linewidth=2)
    
    ax.set_title(num_var)
    ax.set_ylabel(f"Valores de {num_var}")

    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

Los gráficos representados permiten comprobar que las variables numéricas del conjunto de datos presentan valores coherentes con su interpretación, sin evidencias de errores como valores negativos o incoherentes. No obstante, en algunas de estas variables se observan valores extremos, especialmente en aquellas relacionadas con los niveles de inventario y el precio. Estos valores no se consideran errores, sino que reflejan comportamientos reales del negocio, como picos de demanda en determinados productos o artículos con precios significativamente más elevados que la media. Por este motivo, no se contempla la posibilidad de aplicar ningún tratamiento específico sobre estos valores, ya que su eliminación podría suponer la pérdida de información relevante para el entrenamiento de los modelos de predicción.

Por otro lado, las proporciones de las dos categorías de las variables binarias _isPromo_, _bolOpen_ y _bolHoliday_ son coherentes con la naturaleza de los propios atributos. En este caso, en la mayoría de los días del _dataset_ los artículos no se someten a una promoción, la gran mayoría de fechas no se corresponden con días festivos, y el establecimiento abrió sus puertas durante gran parte de los días del conjunto de datos.

Finalmente, la distribución de horizontes de la demanda es muy uniforme y, teniendo en cuenta que la demanda se ofrece para cada producto y una vez al día, indica que los intervalos de las series temporales de la demanda de cada ítem son homogéneos. Asimismo, este periodo abarca dos años, desde marzo del 2024 hasta el mismo mes del año 2026. No obstante, esto entra en conflicto con lo observado en el histograma de la variable _producto_, de manera que se considera la posibilidad de que el _dataset_ no incluya la información de los 1000 productos. Esto se profundizará en el apartado [_producto_](#ej3.1.1) del **Análisis Estadístico** de las variables.

<a id='ej2.2.3'></a>
### 2.2.3 Inconsistencias

El conjunto de datos global del que se dispone contiene las variables _udsVenta_ y _udsStock_, las cuales registran las ventas de un producto en un día concreto, y el volumen de unidades de un ítem concreto en una fecha determinada. El objetivo de esta sección es analizar aquellos casos donde el número de ventas en un día supera el inventario disponible, hecho que es imposible, así como tratar de entender el origen potencial de este fenómeno en el _dataset_. Para ello, se analizan las inconsistencias a nivel general y a nivel de producto.

In [ ]:
# Inconsistencias
inconsistencias: int = dataset[dataset["udsVenta"] > dataset["udsStock"]].shape[0]
print(f"Existen {inconsistencias} casos donde se vendieron más unidades que stock disponible")

In [ ]:
# Inconsistencias por producto
inconsistencias_producto: pd.Series = dataset.groupby("producto", as_index=False).apply(
    lambda g: (g["udsVenta"] > g["udsStock"]).mean() * 100
)
inconsistencias_producto.columns = ["producto", "inconsistencias"]

ax: matplotlib.axes.Axes = inconsistencias_producto.plot(
    kind="scatter",
    x="producto",
    y="inconsistencias",
    alpha=0.6,
    color="blue"
)

ax.set_title("Fracción de inconsistencias por producto")
ax.set_xlabel("producto")
ax.set_ylabel("Inconsistencias (%)")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Los 5 productos con mayor proporción de anomalías
inconsistencias_producto.sort_values(by="inconsistencias", ascending=False).head()

A pesar de que los 1621 casos de inconsistencias suponen una fracción muy reducida del conjunto de datos global, esta observación plantea una cuestión aún más importante, ya que la fuente de las anomalías bien puede ser la variable objetivo _udsVentas_ o el atributo auxiliar _udsStock_, o incluso ambas. Para ello, es importante revisar las ventas e inventarios de algunos de los productos con mayor proporción de anomalías.

In [ ]:
productos: List[int] = (
    inconsistencias_producto
    .sort_values(by="inconsistencias", ascending=False)["producto"]
    .tolist()[:5]
)
fig, axs = plt.subplots(nrows=len(productos), ncols=1, figsize=(7, 20))

for ax, producto in zip(axs, productos):
    dataframe_producto: pd.DataFrame = dataset[dataset["producto"] == producto].copy()
    
    ax.plot(dataframe_producto["fecha"], dataframe_producto["udsStock"], label="Stock", color="blue", alpha=0.6)
    ax.plot(dataframe_producto["fecha"], dataframe_producto["udsVenta"], label="Ventas", color="orange", alpha=0.6)
    
    ax.set_title(f"Inventario y ventas del producto {producto}")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Unidades")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

Tal y como se observa en los gráficos del inventario disponible frente a la demanda, existen muchos casos en los que el _stock_ permanece en valores nulos durante periodos prolongados mientras se registran ventas de forma continuada y uniforme respecto a otros tramos del intervalo. Además, la caída del inventario a 0 en varias ocasiones no se produce paulatinamente, sino de forma abrupta. Adicionalmente, se identifican variaciones en el inventario no explicadas por las ventas, lo que apunta a la existencia de ajustes no observados en el sistema. En un contexto real, estos fenómenos pueden deberse a descartes por controles de calidad periódicos, fechas de caducidad, o accidentes, pero las caídas bruscas del _stock_ en muchos casos resultan complicadas de explicar por estos hechos de los que no se tiene información, por lo que se considera que se dispone de evidencia suficiente para situar el origen de las inconsistencias en _udsStock_ en lugar de _udsVenta_. Asimismo, estas anomalías pueden estar ocurriendo debido a que la variable _stock_ puede no estar siendo correctamente actualizada en determinados periodos. 

En conjunto, estos resultados indican que las inconsistencias detectadas se deben principalmente a limitaciones en la calidad de la variable de inventario, y no a errores en la variable de ventas. Este hecho no supone un problema de cara a la fase de modelado, ya que este atributo no se de debe añadir en los _datasets_ que se usarán para entrenar a los modelos. Consecuentemente, las filas con inconsistencias se mantendrán en el conjunto de datos global ya que se concluye que la fuente de anomalías no es la variable objetivo sino un atributo que no se usará como _input_ en los modelos.

<a id='ej2.2.4'></a>
### 2.2.4 Roturas de _stock_

Dentro del contexto de la previsión de la demanda, uno de los puntos más importantes que se deben tener en cuenta respecto al conjunto de datos que se utilizará para entrenar los modelos gira alrededor de las roturas de _stock_. Este fenómeno se produce cuando no se dispone de suficiente inventario para satisfacer la demanda en una fecha concreta. Este hecho es muy importante a considerar ya que, como bien se ha explicado anteriormente, la variable _udsStock_ no se utilizará como entrada en los modelos predictivos. Consecuentemente, los modelos únicamente tendrán constancia de _udsVenta_, por lo que no podrán saber si los ceros en esta variable son porque la demanda es baja o porque se ha producido una rotura de _stock_. Asimismo, en este último caso el valor de _udsVenta_ no refleja la demanda real ya que, por ejemplo, de haber dispuesto de suficiente inventario se hubieran vendido _n_ unidades de un producto concreto.

In [ ]:
# Casos de demanda baja
demanda_baja: int = dataset[(dataset["udsVenta"] == 0) & (dataset["udsStock"] > 0)].shape[0]
print(f"En {demanda_baja} casos la demanda fue baja")

In [ ]:
# Casos de demanda baja por producto
demanda_baja_producto: pd.Series = dataset.groupby("producto", as_index=False).apply(
    lambda g: ((g["udsVenta"] == 0) & (g["udsStock"] > 0)).mean() * 100
)
demanda_baja_producto.columns = ["producto", "demanda_baja"]

ax: matplotlib.axes.Axes = demanda_baja_producto.plot(
    kind="scatter",
    x="producto",
    y="demanda_baja",
    alpha=0.6,
    color="blue"
)

ax.set_title("Fracción de demanda baja por producto")
ax.set_xlabel("producto")
ax.set_ylabel("Demanda baja (%)")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

En cualquier empresa es de esperar que la demanda de todos los productos no será la misma, donde un pequeño porcentaje de ítems presentará una demanda muy alta y la gran mayoría de productos tendrán una demanda asociada relativamente baja. Esto mismo es lo que se observa en el gráfico _Fracción de demanda baja por producto_, donde la baja demanda se encuentra muy extendida en muchos productos.

In [ ]:
# Potenciales roturas de stock
roturas_stock: int = dataset[(dataset["udsVenta"] == 0) & (dataset["udsStock"] == 0)].shape[0]
print(f"En {roturas_stock} casos se produjeron roturas de inventario")
print(f"Estas roturas representan un {round((roturas_stock / len(dataset) * 100), 2)}% del total de registros del dataset")

In [ ]:
# Roturas de stock por producto
roturas_stock_producto: pd.Series = dataset.groupby("producto", as_index=False).apply(
    lambda g: ((g["udsVenta"] == 0) & (g["udsStock"] == 0)).mean() * 100
)
roturas_stock_producto.columns = ["producto", "roturas_stock"]

ax: matplotlib.axes.Axes = roturas_stock_producto.plot(
    kind="scatter",
    x="producto",
    y="roturas_stock",
    alpha=0.6,
    color="blue"
)

ax.set_title("Fracción de roturas de stock por producto")
ax.set_xlabel("producto")
ax.set_ylabel("Roturas de stock (%)")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot de las roturas de stock por producto
roturas_stock_producto.boxplot(
    column="roturas_stock",
    patch_artist=True,
    boxprops=dict(facecolor="blue", alpha=0.6, color="black", linewidth=1),
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(color="black", linewidth=1),
    capprops=dict(color="black", linewidth=1)
)

plt.axhline(
    roturas_stock_producto["roturas_stock"].mean(),
    color="blue",
    linestyle="--",
    linewidth=2
)

plt.title("Fracción de roturas de stock totales por producto")
plt.ylabel("Roturas de stock (%)")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

Las roturas de _stock_, a diferencia de la demanda baja, es un fenómeno minoritario en la mayoría de productos que se concentra en algunos casos muy específicos. Por lo tanto, la mayoría de ceros en _udsVenta_ se corresponden a la demanda real, pero existe un reducido subconjunto donde esta demanda real se encuentra censurada por la falta de inventario, lo que introduce un sesgo en la variable objetivo y comporta su infraestimación en estos casos.

Así pues, se considera que la manera adecuada de proceder es mediante un proceso de imputación de los valores de la variable _udsVenta_ que se corresponden con una rotura de _stock_. Este procedimiento debe realizarse con cautela, ya que se está imputando la propia variable objetivo, y la distribución de los _stockouts_ no es homogénea en todos los productos como se muestra en el diagrama de la caja.
	
Para facilitar este proceso, se decide descartar del conjunto de datos aquellos productos cuyo porcentaje de _stockouts_ supera el 10%, dado que únicamente representan aproximadamente un 4% del total de artículos. Esta decisión se fundamenta en que, en estos casos, la imputación de la variable objetivo implicaría reconstruir una parte significativa de la serie temporal, lo que podría introducir un sesgo considerable y distorsionar los patrones reales de la demanda. Por el contrario, para productos con una baja proporción de roturas, la imputación afecta únicamente a una fracción reducida de las observaciones, por lo que el impacto sobre la distribución de la variable objetivo es limitado. Asimismo, establecer un umbral inferior al 10% se considera excesivamente restrictivo, ya que conllevaría la eliminación de un número elevado de productos y, en consecuencia, una pérdida relevante de información en el conjunto

In [ ]:
# Productos con un porcentaje de rotura de stock superior al 10%
descartes: int = (
    roturas_stock_producto[roturas_stock_producto["roturas_stock"] > 10]
    .sort_values(by="roturas_stock", ascending=False)
    .shape[0]
)

print(
    f"Existen {descartes} productos con un porcentaje de rotura de stock superior al 10%, que representan un "
    f"{round(100 * (descartes / roturas_stock_producto.shape[0]), 2)}% del total de ítems"
)

<br><br>Dicho esto, el proceso de imputación que se seguirá es una interpolación lineal de la variable _udsVenta_, ya que permite estimar los valores faltantes a partir de la información local de la serie, preservando su estructura temporal. Esta metodología proporciona una aproximación razonable de la demanda sin introducir una complejidad innecesaria ni supuestos difíciles de justificar.

In [ ]:
# Se guarda la variable udsVenta original para compararla con ella misma tras la imputación
dataset["udsVenta_original"] = dataset["udsVenta"]

In [ ]:
# Interpolación de udsVenta en stockouts
dataset.loc[(dataset["udsVenta"] == 0) & (dataset["udsStock"] == 0), "udsVenta"] = np.nan

dataset["udsVenta"] = dataset.groupby("producto")["udsVenta"].transform(
    lambda g: g.interpolate(method="linear")
)

In [ ]:
# Función para hacer un gráfico comparativo de las ventas tras la imputación de udsVenta
def grafico_ventas(productos: List[int], df: pd.DataFrame) -> None:
    """Se crea un gráfico de las ventas de los productos antes y después del proceso de imputación

    Argumentos:
        productos: List[int] -> Lista con las IDs de los productos
        df: pd.DataFrame -> DataFrame de pandas con los datos de los productos

    Devuelve:
        None
    """
    fig, axs = plt.subplots(nrows=len(productos), ncols=1, figsize=(7, 20))
    
    for ax, producto in zip(axs, productos):
        dataframe_producto: pd.DataFrame = df[dataset["producto"] == producto]
    
        ax.plot(dataframe_producto["fecha"], dataframe_producto["udsVenta"], label="Ventas imputadas", color="red", alpha=0.6)
        ax.plot(dataframe_producto["fecha"], dataframe_producto["udsVenta_original"], label="Ventas originales", color="blue", alpha=0.5)
    
        ax.set_title(f"udsVentas del producto {producto} antes y después de la imputación")
        ax.set_xlabel("Fecha")
        ax.set_ylabel("Unidades")
        ax.legend(loc="upper left")
        ax.grid(True, alpha=0.3)
        ax.set_axisbelow(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Comparación de udsVenta de los 5 productos con más porcentaje de stockout pero menor del 10%
productos: List[int] = (
    roturas_stock_producto[roturas_stock_producto["roturas_stock"] <= 10]
    .sort_values(by="roturas_stock", ascending=False)["producto"]
    .tolist()[:5]
)

grafico_ventas(productos, dataset)

In [ ]:
# Comparación de udsVenta de los 5 productos con más porcentaje de stockout
productos: List[int] = (
    roturas_stock_producto
    .sort_values(by="roturas_stock", ascending=False)["producto"]
    .tolist()[:5]
)

grafico_ventas(productos, dataset)

In [ ]:
# Dataset con los productos con una proporción de stockout inferior al 10%
dataset: pd.DataFrame = dataset.merge(roturas_stock_producto, on="producto", how="left")
dataset_filtrado: pd.DataFrame = dataset[dataset["roturas_stock"] <= 10]

La validación visual de la imputación pone de manifiesto diferencias significativas en función de la proporción de las roturas de _stock_. En aquellos productos con la proporción de _stockout_ más alta de los que tienen una fracción inferior al umbral del 10%, la imputación mediante interpolación permite reconstruir la demanda de forma coherente, respetando la dinámica de la serie temporal en intervalos de duración breve. Sin embargo, en aquellos productos con una elevada proporción de roturas de _stock_, la falta de información fiable provoca que la imputación genere patrones artificiales, como tendencias lineales prolongadas que no reflejan el comportamiento real de la demanda. Este resultado refuerza la necesidad de excluir del análisis aquellos productos con una proporción elevada de roturas de _stock_, dado que la imputación en estos casos introduce un sesgo significativo.

Por otro lado, cabe considerar los casos extremos donde el primer registro de la demanda histórica de un producto se corresponde a un _stockout_. En estas situaciones no se pueden imputar valores de forma lineal ya que no se dispone de registros anteriores para hacer la interpolación, por lo que siguen codificados como _NaN_. Asimismo, los días consiguientes a estas fechas también siguen conteniendo valores nulos hasta que ya no hay rotura de inventario.

In [ ]:
nulos_al_principio: pd.DataFrame = dataset_filtrado.groupby("producto", as_index=False).agg(
    nulos=("udsVenta", lambda g: g.isna().sum()),
    nulos_porcentaje=("udsVenta", lambda g: g.isna().mean() * 100)
)
nulos_al_principio: pd.DataFrame = (
    nulos_al_principio[nulos_al_principio["nulos"] > 0]
    .sort_values(by="nulos")
)

print(f"Hay {nulos_al_principio.shape[0]} productos con valores nulos en udsVenta al principio de la serie temporal")

In [ ]:
# Valor absoluto de nulos
ax: matplotlib.axes.Axes = nulos_al_principio.plot(
    kind="bar",
    x="producto",
    y="nulos",
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

ax.set_title(f"Valores nulos al principio de la serie temporal por producto")
ax.set_xlabel("producto")
ax.set_ylabel("Valores nulos")

ax.tick_params(axis="x", rotation=45)
ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)
ax.get_legend().remove()
    
plt.tight_layout()
plt.show()

In [ ]:
# Porcentaje de nulos
ax: matplotlib.axes.Axes = nulos_al_principio.plot(
    kind="bar",
    x="producto",
    y="nulos_porcentaje",
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

ax.set_title(f"Fracción de nulos al principio de la serie temporal por producto")
ax.set_xlabel("producto")
ax.set_ylabel("Porcentaje de nulos")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)
ax.get_legend().remove()
    
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot del valor absoluto de valores nulos
ax: matplotlib.axes.Axes = nulos_al_principio["nulos_porcentaje"].plot(
    kind="box",
    patch_artist=True,
    boxprops=dict(facecolor="blue", alpha=0.6, color="black", linewidth=1),
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(color="black", linewidth=1),
    capprops=dict(color="black", linewidth=1)
)

plt.axhline(
    nulos_al_principio["nulos_porcentaje"].mean(),
    color="blue",
    linestyle="--",
    linewidth=2
)

ax.set_title("Stockouts iniciales")
ax.set_ylabel("Roturas de stock (%)")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)
    
plt.tight_layout()
plt.show()

De la misma forma que con los _stockouts_, la distribución de valores nulos al principio de las series temporales no es uniforme, de manera que unos pocos productos contienen menos de un 2% de valores faltantes al inicio de la serie, y otros pocos entre un 2 y un 6%. Dado que estos valores deben ser imputados sin disponer de información previa en la serie, su tratamiento introduce un mayor grado de incertidumbre en comparación con la imputación de _udsVenta_ cuando ocurren roturas de inventario en mitad de la serie temporal. Por otro lado, dado que las series analizadas presentan un horizonte amplio de dos años, el impacto de la imputación en los valores iniciales resulta limitado. En particular, los valores imputados representan una fracción reducida de la serie y se sitúan en el extremo inicial, alejados del horizonte de predicción, por lo que su influencia sobre los modelos de _forecasting_ es marginal, ya que estos se apoyan principalmente en la dinámica más reciente de la serie temporal. Consecuentemente, se procede con la eliminación de estos valores nulos iniciales en todos los productos que los presenten. Asimismo, también podrían imputarse estos valores con un método de _backwards fill_ que asigna el primer valor no nulo de la serie a los iniciales, pero en este caso se opta por no introducir un sesgo artificial, por pequeño que sea, porque representaría un segundo proceso de imputación en la variable objetivo.

In [ ]:
# Borrado de los registros con valores nulos al inicio de las series temporales
dataset_filtrado.dropna(inplace=True)
print(f"Tras el tratamiento de las roturas de stock, el dataset contiene {dataset_filtrado.shape[0]} registros")

<br><br><a id='ej3'></a>
# 3. Análisis de las variables

Tras haber evaluado la calidad de los datos y haber aplicado las operaciones pertinentes, se procede con el análisis de las variables del conjunto de datos. En particular, se pondrá el foco sobre aquellas que se empleen en los modelos de aprendizaje automático, por lo que _udsStock_ no se analizará al detalle en un subapartado de dedicado.

<a id='ej3.1'></a>
## 3.1. Análisis de las variables auxiliares

En este apartado se analizan las variables del _dataset_ que ayudan a explicar el comportamiento del atributo objetivo. Concretamente, se focaliza en las variables _producto_, _idSecuencia_, _isPromo_, _bolOpen_, _bolHoliday_, _diasEntrePedidos_, _diasLeadtime_, y _eurPrecioMedio_.

<a id='ej3.1.1'></a>
### 3.1.1. _producto_

El conjunto de datos inicialmente contaba con la información de 1000 productos distintos, y a medida que se ha avanzado en los apartados anteriores, algunos ítems se han ido descartando por su alto contenido en _stockouts_.

In [ ]:
# Productos descartados totales
descartes_totales: int = len(dataset["producto"].unique()) - len(dataset_filtrado["producto"].unique())
print(f"Tras revisar los stockouts, se han descartado un total de {descartes_totales} productos")

In [ ]:
# Total de productos en el dataset actual
productos: int = len(dataset_filtrado["producto"].unique())
print(f"El conjunto de datos contiene información de {productos} productos distintos")

In [ ]:
# Productos faltantes
TOTAL_PRODUCTOS: int = 1000
productos_faltantes: int = TOTAL_PRODUCTOS - productos - descartes_totales
print(f"Inicialmente, el dataset carecía de la demanda de {productos_faltantes} productos")

A pesar de haberse descartado 39 productos en las secciones anteriores, la suma de estos elementos con los actuales es inferior a 1000, lo que indica que el conjunto de datos inicialmente no contenía la demanda histórica de todos los productos. Esto sucede a pesar de que los archivos _DatosCicloAprovisionamiento.xlsx_ y _DatosPrecioMedio.xlsx_ contienen la información correpondiente de cada uno de los 1000 productos. Los datos de los ítems faltantes no se han adherido al dataset global dado que la operación que se ha empleado para unir las tablas de datos al principio era de tipo _**LEFT JOIN**_, situando a la izquierda la demanda histórica. No obstante, esta observación no tiene un impacto negativo de cara a la etapa de modelado, ya que simplemente algunos productos no se tendrán en cuenta en esa fase, lo cual debe tenerse en cuenta a la hora de tomar conclusiones.

In [ ]:
# Distribución de los horizontes de la demanda
ax: matplotlib.axes.Axes = dataset_filtrado.groupby("producto").size().hist(
    log=True,
    bins=10,
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

ax.set_title("Distribución de los horizontes de la demanda en escala logarítmica")
ax.set_xlabel("Duración de los intervalos (días)")
ax.set_ylabel("log(Frecuencia)")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

Por otro lado, el histograma logarítmico del horizonte temporal de cada producto sugiere que, inicialmente, prácticamente todos los productos contaban con el mismo número de fechas en su demanda histórica. Sin embargo, tras haber eliminado los _stockouts_ del comienzo de las series temporales de algunos productos en el apartado [Roturas de _stock_](#ej2.2.3), esto ha dado lugar a una distribución de la duración de los intervalos de la demanda donde, como máximo, las series de algunos productos contienen menos de 700 días. No obstante, en estos casos se sigue disponiendo de una cantidad adecuada de información para entrenar los modelos, y no se considera que esta heterogeneidad sea suficiente como para tener una influencia significativa en el rendimiento de los modelos en estos productos concretos. Por lo tanto, esta observación refuerza la decisión tomada en la sección [Roturas de _stock_](#ej2.2.3), donde se ha considerado que el borrado de los registros de roturas de inventario al principio de las series temporales no introduciría un sesgo importante debido a la amplitud de los horizontes temporales.

<a id='ej3.1.2'></a>
### 3.1.2. _idSecuencia_

Este atributo equivale a las fechas de las que se dispone la demanda de los ítems de la empresa. Entonces, uno de los supuestos que se hace al trabajar con estos datos es que, para cada producto, se debería contar con la demanda a nivel de día, sin huecos temporales entre medio.

In [ ]:
# Función de agregación personalizada para comprobar el espaciado en el tiempo de las fechas
def diferencia_entre_dias(grupo: pd.Series) -> bool:
    """Comprueba si una secuencia de fechas está espaciada en el tiempo por un día en todos
    sus registros

    Argumentos:
        grupo: pd.Series -> Grupo al que aplicar la función tras hacer un groupby()

    Devuelve:
        bool -> Si las fechas del grupo están todas espaciadas un día
    """
    secuencia: pd.Series = grupo.sort_values()
    return (secuencia.diff().dropna().dt.days == 1).all()

In [ ]:
# Comprobación de la granularidad diaria de los datos por producto
intervalos: pd.DataFrame = dataset_filtrado.groupby("producto", as_index=False).agg(
    fecha_inicio=("idSecuencia", "min"),
    fecha_final=("idSecuencia", "max"),
    secuencia_correcta=("fecha", diferencia_entre_dias)
)

if intervalos["secuencia_correcta"].all():
    print("Todas las fechas de todos los productos están espaciadas un día en el calendario")

Por otra banda, una de las observaciones que ya se ha hecho durante el presente análisis exploratorio es que el horizonte temporal de la demanda abraca apróximadamente dos años para todos los ítems. Asimismo, el hecho que todas las fechas estén espaciadas un día indica que todas las series temporales terminan el mismo día en el conjunto de datos con el que es está trabajando. Sin embargo, la eliminación de algunos registros iniciales en [Roturas de _stock_](#ej2.2.3) ha provocado que no todas las series temporales empiecen el mismo día.

In [ ]:
# Histograma de las fechas iniciales de las series temporales
ax: matplotlib.axes.Axes = dataset_filtrado.groupby("producto")["fecha"].agg("min").hist(
    log=True,
    bins=10,
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

ax.set_title("Histograma de las fechas iniciales de la demanda en escala logarítmica")
ax.set_xlabel("fecha")
ax.set_ylabel("log(Frecuencia)")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

La distribución logarítmica de las fechas iniciales de la demanda sitúa buena parte de estos factores cerca del comienzo de la mayoría de las series temporales del _dataset_, siendo este el día 2024-03-22. Asimismo, existe un reducido subconjunto de productos con una inicio ligeramente más alejado, situado como máximo a poco más de un mes del día 2024-03-22, el cual se corresponde con aquellos productos que presentaban una secuencia de _stockouts_ iniciales más duradera. Nótese que se ha usado la variable _fecha_ en el histograma, pero esta es equivalente a _idSecuencia_ y más sencilla de interpretar.

Por último, se añaden al conjunto de datos una serie de variables adicionales relacionadas con la fecha que pueden ser útiles más adelante para relacionar las ventas con patrones temporales de carácter semanal o mensual, siendo estas los días de la semana y los meses.

In [ ]:
# Días de la semana
traducciones: Dict[str, str] = {
    "Monday": "Lunes",
    "Tuesday": "Martes",
    "Wednesday": "Miércoles",
    "Thursday": "Jueves",
    "Friday": "Viernes",
    "Saturday": "Sábado",
    "Sunday": "Domingo"
}

dataset_filtrado["dia_semana_str"] = dataset_filtrado["fecha"].dt.day_name()
dataset_filtrado.replace({"dia_semana_str": traducciones}, inplace=True)
dataset_filtrado["dia_semana_str"] = pd.Categorical(
    dataset_filtrado["dia_semana_str"],
    categories=traducciones.values(),
    ordered=True
)

dataset_filtrado["dia_semana_int"] = dataset_filtrado["fecha"].dt.weekday

In [ ]:
# Meses
dataset_filtrado["mes_int"] = dataset_filtrado["fecha"].dt.month

<a id='ej3.1.3'></a>
### 3.1.3. _idPromo_

La aplicación de promociones generalmente es una buena estrategia para aumentar las ventas de ciertos ítems. Consecuentemente, es un factor que juega un papel importante en la predicción de la demanda. Este atributo es de tipo _booleano_ a pesar de que se codifique como un entero, y debe analizarse tanto de forma global como a nivel de producto.

In [ ]:
# Análisis global de isPromo
proporcion_ispromo: List[float] = [
    round(dataset_filtrado[dataset_filtrado["isPromo"] == 0].shape[0] / dataset_filtrado.shape[0], 2),
    round(dataset_filtrado[dataset_filtrado["isPromo"] == 1].shape[0] / dataset_filtrado.shape[0], 2),
]

plt.bar(
    ["Sin promoción", "En promoción"],
    proporcion_ispromo,
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

plt.title("Distribución de isPromo")
plt.ylabel("Proporción (%)")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Productos sin promociones
promociones_analisis: pd.DataFrame = dataset_filtrado.groupby("producto", as_index=False)["isPromo"].agg("mean")
promociones_analisis.loc[promociones_analisis["isPromo"] > 0, "isPromo"] = 1

plt.bar(
    ["Productos sin ninguna promoción", "Productos con al menos una promoción"], [
        promociones_analisis[promociones_analisis["isPromo"] == 0].shape[0] / promociones_analisis.shape[0],
        promociones_analisis[promociones_analisis["isPromo"] == 1].shape[0] / promociones_analisis.shape[0]
    ], color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

plt.title("Proporción de productos con al menos una promoción")
plt.ylabel("Proporción (%)")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

El gráfico _Distribución de isPromo_ denota que, durante la gran mayoría de días del intervalo de las demandas históricas, a los productos no se les aplican promociones, siendo estas situaciones casos excepcionales. Esta observación se ve reforzada por el hecho que a prácticamente la mitad de los productos no se les aplican promociones en ningún momento de sus historial de la demanda. La aplicación de promociones pues, es un caso aislado que puede deberse a distintos factores, como períodos vacacionales donde la demanda suele crecer y se busca aumentar las ventas mediante promociones, liquidaciones, o simplemente se desea aumantar las ventas de los ítems por cualquier otra razón. No obstante, esto es a nivel general en el _dataset_, pero hace falta realizar el análisis también a nivel de producto.

In [ ]:
# Análisis por producto de isPromo
analisis_promo: pd.DataFrame = dataset_filtrado.groupby("producto").agg(
    pct_promo=("isPromo", "mean"),
    promedio_ventas=("udsVenta", "mean"),
    promedio_ventas_promo=("udsVenta", lambda g: g[dataset_filtrado.loc[g.index, "isPromo"] == 1].mean()),
    promedio_ventas_no_promo=("udsVenta", lambda g: g[dataset_filtrado.loc[g.index, "isPromo"] == 0].mean())
).reset_index()

In [ ]:
# Porcentajes de promoción en el intervalo temporal de cada producto
plt.bar(
    analisis_promo["producto"],
    analisis_promo["pct_promo"],
    color="blue",
    alpha=0.6
)

plt.title("Duración porcentual de las promociones por producto")
plt.xlabel("producto")
plt.ylabel("pct_promo")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de los porcentajes de promoción en la duración de los intervalos de la demanda
ax: matplotlib.axes.Axes = analisis_promo["pct_promo"].hist(
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

ax.set_title("Distribución de la duración porcentual de las promociones por producto")
ax.set_xlabel("pct_promo")
ax.set_ylabel("Frecuencia")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)
    
plt.tight_layout()
plt.show()

In [ ]:
# Productos con más de un 80% de tiempo promocional
print(
    f"Hay un total de {analisis_promo[analisis_promo["pct_promo"] >= 0.8].shape[0]} productos con "
    "más de un 80% de tiempo promocional"
)
print(
    f"Algunos de estos productos son los de IDs {
        ", ".join([str(n )for n in analisis_promo[analisis_promo["pct_promo"] >= 0.8].sort_values(by="pct_promo")["producto"].tolist()[:5]])
    }"
)

De la misma manera que con el conjunto de datos a nivel general, la mayoría de productos están sujetos a períodos promocionales muy breves, mientras que un reducido subconjunto de ítems cuentan con promociones muy duraderas, hasta el punto que en varios casos abarcan más del 80% del intervalo de demanda del que se dispone. Esta última observación puede deberse a varios factores como los mencionados anteriormente. 

Por otro lado, para profundizar más en el análisis a nivel de producto, se añade la variable _mejora_promo_, que es la diferencia entre _promedio_ventas_promo_ y _promedio_ventas_no_promo_, la cual indica si una promoción resulta en más ventas de un producto concreto.

In [ ]:
analisis_promo["mejora_promo"] = analisis_promo["promedio_ventas_promo"] - analisis_promo["promedio_ventas_no_promo"]

In [ ]:
# Efectividad de las promociones en las ventas
plt.bar(
    analisis_promo["producto"],
    analisis_promo["mejora_promo"],
    color="blue",
    alpha=0.6
)

plt.title("Efectividad de las promociones por producto")
plt.xlabel("producto")
plt.ylabel("mejora_promo")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de la efectividad de las promociones en las ventass
ax: matplotlib.axes.Axes = analisis_promo["mejora_promo"].hist(
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

ax.set_title("Distribución de la efectividad de las promociones por producto")
ax.set_xlabel("mejora_promo")
ax.set_ylabel("Frecuencia")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# IDs de los ítems cuyas ventas mejoran más con las promociones, e IDs de los que las
# ventas empeoran
productos_mejora: List[int] = analisis_promo.sort_values(by="mejora_promo", ascending=False)["producto"].tolist()[:5]
productos_empeora: List[int] = analisis_promo.sort_values(by="mejora_promo")["producto"].tolist()[:5]

print(f"Las ventas de los productos {", ".join([str(n) for n in productos_mejora])} mejoran significativamente cuando se les aplica una promoción")
print(f"Las ventas de los ítems {", ".join([str(n) for n in productos_empeora])} empeoran de forma considerable cuando se les aplica una promoción")

Gracias a la variable compuesta _mejora_promo_, se puede observar cómo la mayoría de productos permanecen prácticamente invariantes cuando se les aplica una promoción. Sin embargo, también se aprecia un mayor volumen de productos cuyas ventas mejoran cuando están sujetos a promociones que en el caso contrario, es decir, cuyas ventas empeoran. Esto es de esperar, ya que es un comportamiento intuitivo cuando un producto forma parte de una promoción, pero también cabe destacar que hay una cantidad considerable cuyas ventas disminuyen cuando se promocionan. Una posible explicación de este hecho es que la aplicación de una promoción a un cierto conjunto de ítems puede generar un efecto de competencia interna entre productos, de manera que el incremento de ventas en algunos artículos se produce a costa de otros también promocionados. Este fenómeno puede estar motivado por la limitada atención del consumidor o por restricciones presupuestarias, que llevan al cliente a priorizar unos productos frente a otros dentro del mismo contexto promocional. Asimismo, cabe recordar que la empresa en cuestión distribuye productos para el automóvil, lo cual introduce una gran heterogeneidad en la naturaleza de la demanda. Estos ítems pueden abarcar desde productos muy específicos, cuya compra está condicionada por necesidades puntuales y poco frecuentes como averías o sustituciones concretas, hasta productos más genéricos o impulsivos, como ambientadores o accesorios. En este contexto, es más probable que los efectos negativos de la promoción se produzcan en aquellos productos con mayor grado de sustituibilidad, donde el consumidor dispone de múltiples alternativas.

In [ ]:
# Impacto de las promociones
plt.scatter(
    analisis_promo["pct_promo"],
    analisis_promo["mejora_promo"],
    color="blue",
    alpha=0.6
)

plt.title("Impacto de las promociones por producto")
plt.xlabel("pct_promo")
plt.ylabel("mejora_promo")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

El gráfico _Impacto de las promociones por producto_ pone de manifiesto que aumentar la duración de las ofertas no necesariamente aporta más ventas, ya que buena parte de los ítems con más de un 80% de tiempo promocional presentan menos ventas exceptuando un par de casos donde las ventas sí mejoran considerablemente. Por el contrario, los ítems con períodos promocionales de menos del 30% son los que sí exhiben mejoras muy importantes en las ventas, lo cual puede deberse a que estos intervalos más cortos sirven para que el comprador pueda apreciar las diferencias de precio de los productos cuando estos se promocionan, mientras que cuando este tiempo es mucho más largo quizás el comprador acaba interiorizando el precio de estos elementos como el precio no promocional, olvidando la noción de cuánto costaba antes de la oferta. No obstante, esto depende de la naturaleza del producto en cuestión ya que, como se mencionó anteriormente, el cliente pueda percatarse de este hecho solamente en los ítems de menor necesidad inmediata, mientras que en los más urgentes la compra se realiza sin tener demasiado en cuenta los precios. Por otro lado, este gráfico también ilustra que las promociones se aplican a conjuntos de productos, ya que pueden observarse agrupaciones de los ítems para un mismo período de oferta. Este hecho puede deberse a una estrategia propia de la empresa, donde se dispone de una clasificación de productos similares en base a sus ventas a los que se aplica el mismo período promocional. Consecuentemente, esto puede resultar en que dos productos totalmente distintos pueden considerarse iguales en cuanto a sus ventas para planificar las ofertas de forma más sencilla, por lotes de ítems en lugar de planearse individualmente.

In [ ]:
# Intensidad de las promociones por fecha
dataset_filtrado["año_mes"] = dataset_filtrado["fecha"].dt.to_period("M")
dataset_filtrado["dia"] = dataset_filtrado["fecha"].dt.day

datos_heatmap: pd.DataFrame = dataset_filtrado.groupby(["año_mes", "dia"])["isPromo"].mean().unstack()

plt.figure(figsize=(10, 8))

sns.heatmap(datos_heatmap, cmap="Reds")

plt.title("Intensidad de promociones por día y mes")
plt.xlabel("Día del mes")
plt.ylabel("Mes")

plt.tight_layout()
plt.show()

In [ ]:
# Promociones por día de la semana
promo_semanal: pd.DataFrame = dataset_filtrado.groupby("dia_semana_str")["isPromo"].mean()

promo_semanal.plot(
    kind="bar",
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

plt.title("Promociones por día de la semana")
plt.xlabel("Días de la semana")
plt.tick_params(axis="x", rotation=45)
plt.ylabel("Días con promoción (%)")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

La intensidad de promociones a lo largo del calendario muestra que, si bien no existe una estacionalidad extremadamente marcada, sí se aprecian ciertos periodos con mayor concentración de campañas promocionales. En particular, algunos meses presentan niveles más elevados de intensidad, lo que sugiere la existencia de estrategias comerciales puntuales más que un patrón estrictamente estacional. Este comportamiento es coherente con la naturaleza de la empresa, dedicada a la distribución de productos para el automóvil. A diferencia de otros sectores más sensibles a la estacionalidad, como la moda o el turismo, la demanda de muchos de estos productos está ligada a necesidades funcionales o imprevistas , lo que reduce la dependencia directa de la época del año. No obstante, sí pueden identificarse ciertos incrementos en la actividad promocional en momentos concretos, posiblemente asociados a campañas comerciales específicas o a productos de carácter más discrecional, como accesorios o elementos decorativos del vehículo. En estos casos, la estacionalidad puede tener un mayor peso, especialmente en periodos como verano y Navidad o fechas cercanas a campañas como el _black friday_ en noviembre. 

Asimismo, el análisis de la distribución de promociones a lo largo de la semana muestra que su intensidad se mantiene prácticamente constante entre los distintos días, sin apreciarse diferencias significativas entre días laborables y fines de semana. Este resultado refuerza la idea de que las promociones en este contexto no están orientadas a maximizar el impacto en momentos específicos de consumo, sino que responden a estrategias de mayor duración en el tiempo.

Así pues, en conjunto, el patrón observado sugiere que las promociones en este sector responden en mayor medida a decisiones estratégicas de la empresa que a una estacionalidad estricta de la demanda, aunque ciertos segmentos de producto sí pueden presentar comportamientos más sensibles al calendario.

<a id='ej3.1.4'></a>
### 3.1.4. _bolOpen_

El estudio de la variable _bolOpen_ resulta interesante en este contexto, ya que los horarios de apertura de una tienda de productos del automóvil pueden diferir significativamente respecto a otro tipo de comercios. Esto se debe a que algunos ítems que de los establecimientos de la empresa presente pueden ser de alta urgencia, lo que posiblemente provoque un horario de apertura más amplio a lo largo de la semana.

In [ ]:
# Distribución de aperturas de la tienda durante los días de la semana
apertura_semanal: pd.DataFrame = dataset_filtrado.groupby("dia_semana_str")["bolOpen"].mean()

apertura_semanal.plot(
    kind="bar",
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

plt.title("Distribución de las aperturas de la tienda durante los días de la semana")
plt.xlabel("Días de la semana")
plt.tick_params(axis="x", rotation=45)
plt.ylabel("Aperturas (%)")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Matriz de aperturas
calendario_aperturas: pd.DataFrame = dataset_filtrado.groupby("idSecuencia").agg({
    "año_mes": "first",
    "dia": "first",
    "bolOpen": "first",
    "dia_semana_str": "first",
    "bolHoliday": "first"
})

datos: pd.DataFrame = calendario_aperturas.pivot(
    index="año_mes",
    columns="dia",
    values="bolOpen"
).fillna(0)

# Gráfico del calendario de aperturas de la tienda
fig, ax = plt.subplots(figsize=(10, 8))
cmap: ListedColormap = ListedColormap(["white", "black"])

im = ax.imshow(datos.values, cmap=cmap, aspect="auto")

ax.set_xticks(np.arange(datos.shape[1]))
ax.set_yticks(np.arange(datos.shape[0]))

ax.set_xticklabels(datos.columns)
ax.set_yticklabels(datos.index)

ax.set_xlabel("Día del mes")
ax.set_ylabel("Mes")
ax.set_title("Calendario de aperturas")

ax.set_xticks(np.arange(-.5, datos.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-.5, datos.shape[0], 1), minor=True)

# Se marcan los domingos en rojo
domingos: pd.DataFrame = calendario_aperturas[calendario_aperturas["dia_semana_str"] == "Domingo"]

for _, row in domingos.iterrows():
    fila: int = list(datos.index).index(row["año_mes"])
    col: int = list(datos.columns).index(row["dia"])

    ax.add_patch(plt.Rectangle(
        (col - 0.5, fila - 0.5),
        1, 1,
        fill=False,
        edgecolor="red",
        linewidth=2
    ))

plt.tight_layout()
plt.show()

La distribución de aperturas a lo largo de los días de la semana del establecimiento indica que la tienda, generalmente, sigue un horario de lunes a sábado, ofreciendo sus servicios durante parte del fin de semana. Además, el gráfico demuestra que el local también ha abierto sus puertas prácticamente la mitad de todos los domingos de los intervalos de la demanda. Esta observación se complementa con el calendario de aperturas de la tienda, donde se aprecia que el establecimiento permaneció cerrado durante aproximadamente todos los domingos del año 2024 hasta mitades del año 2025, pero a partir de este punto el local abrió sus puertas a diario prácticamente. Además, también resulta interesante comprobar si la tienda también permaneció abierta en este intervalo de tiempo durante las festividades, aunque esto se realizará en el apartado [_bolHoliday_](#ej3.1.5). Volviendo a la cuestión, esta observación puede deberse a una decisión empresarial para aumentar las ventas, o a una nueva estrategia donde se hayan repartido las horas de apertura semanales de tal forma que el local pueda abrir los domingos sin aumentar el volumen total de tiempo de apertura semanal.

<a id='ej3.1.5'></a>
### 3.1.5. _bolHoliday_

In [ ]:
# Calendario de festivos y aperturas de la tienda
fig, ax = plt.subplots(figsize=(10, 8))
cmap: ListedColormap = ListedColormap(["white", "black"])

im = ax.imshow(datos.values, cmap=cmap, aspect="auto")

ax.set_xticks(np.arange(datos.shape[1]))
ax.set_yticks(np.arange(datos.shape[0]))

ax.set_xticklabels(datos.columns)
ax.set_yticklabels(datos.index)

ax.set_xlabel("Día del mes")
ax.set_ylabel("Mes")
ax.set_title("Calendario de festivos y aperturas")

ax.set_xticks(np.arange(-.5, datos.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-.5, datos.shape[0], 1), minor=True)

# Se marcan los domingos en rojo
domingos: pd.DataFrame = calendario_aperturas[calendario_aperturas["dia_semana_str"] == "Domingo"]

for _, row in domingos.iterrows():
    fila: int = list(datos.index).index(row["año_mes"])
    col: int = list(datos.columns).index(row["dia"])

    ax.add_patch(plt.Rectangle(
        (col - 0.5, fila - 0.5),
        1, 1,
        fill=False,
        edgecolor="red",
        linewidth=2
    ))

# Se añaden las festividades con un cuadrado azul 
festividades: pd.DataFrame = calendario_aperturas[calendario_aperturas["bolHoliday"] == 1]

for _, row in festividades.iterrows():
    fila: int = list(datos.index).index(row["año_mes"])
    col: int = list(datos.columns).index(row["dia"])

    ax.text(
        col, fila,
        "■",
        ha="center",
        va="center",
        color="blue",
        fontsize=10
    )

plt.tight_layout()
plt.show()

La distribución de los días festivos observada en el _dataset_ es coherente con calendarios laborales catalanes, donde los días de la diada catalana, el lunes de Pascua y San Esteban se consideran días festivos donde la empresa no abre. Por otro lado, en el apartado anterior se observó que el local empezó a abrir cada día a partir de mitad del año 2025, lo que se consideró como una posible estrategia empresarial. No obstante, la adición de _bolHoliday_ en el calendario revela inconsistencias a partir de mediados de este mismo período, donde también desaparecen los días festivos. Este comportamiento resulta irreal desde un punto de vista operativo, lo que sugiere la existencia de datos incompletos o errores en el registro. Consecuentemente, se opta por reconstruir la información relacionada tanto con las festividades como con las aperturas, hecho que permite disponer de variables coherentes a lo largo de todo el horizonte temporal, evitando la pérdida de datos y reduciendo el impacto de posibles errores en el _dataset_ original.

Para realizar esta tarea debe extremarse la cautela y se opta por un proceso manual donde, mediante la librería _holidays_ de _Python_, se anotan las festividades del año 2024 en Cataluña. Una vez se dispone de los festivos del año 2024, se comprueba mediante el gráfico _Calendario de aperturas_ si la empresa reconoce todas estas festividades y se examina si estos días abrieron la tienda, lo cual permitirá reconstruir _bolOpen_. Tras este análisis, cabe destacar que la empresa no reconoce como festivos ni el día de la constitución española ni el día de la Inmaculada Concepción. Por otro lado, la distribuidora de productos de automóvil sí considera como festivo el día 20-05-2024, que no mostró la librería _holidays_, que se corresponde con la segunda Pascua que solo se celebró en algunas ciudades concretas como Barcelona y otros municipios. Adicionalmente, se ha identificado un día que no se asocia a ninguna festividad que se marca como día vacacional, siendo este el 31-05-2024, el cual se considera como día festivo propio de la empresa. Por último, como la demanda histórica comienza a partir de marzo, no se dispone de la información referente a la apertura del establecimiento en las fechas de año nuevo y reyes magos.

In [ ]:
# Resumen de días festivos en el año 2024 de la empresa
festivos_2024: pd.DataFrame = pd.DataFrame({
    "Días": [
        "2024-01-01", "2024-01-06", "2024-03-29", "2024-04-01", "2024-05-01", "2024-05-20", "2024-05-31", "2024-06-24",
        "2024-08-15", "2024-09-11", "2024-10-12", "2024-11-01", "2024-12-06", "2024-12-08", "2024-12-25", "2024-12-26"
    ],
    "Festividades": [
        "Año nuevo", "Día de reyes", "Viernes santo", "Lunes de Pascua", "Día del trabajador", "Segunda Pascua",
        "Día propio de la empresa", "San Juan", "Día de la Asunción de la Virgen María", "Diada de Cataluña",
        "Fiesta nacional de España", "Todos los Santos", "Día de la constitución española", "Día de la Inmaculada Concepción",
        "Navidad", "San Esteban"
    ],
    "¿Se considera festivo?": [np.nan, np.nan, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1],
    "¿Se abrió?": [np.nan, np.nan, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0]
})

festivos_2024.head(festivos_2024.shape[0])

Este proceso se repite para el año 2025, en el cual se observa que, a partir de enero, las únicas festividades marcadas son los domingos, ignorando tanto semana santa como las festividades de mayo y junio, de las que se tiene constancia que la empresa celebra como se demostró en 2024. Así pues, las únicas festividades reflejadas en el calendario del año 2025 son la de año nuevo y reyes magos, que estaban ausentes en 2024, de manera que se dispone de los días que la empresa considera como festivos en el lapso de un año completo. Dicho esto, los períodos vacacionales de la empresa en 2025 y las aperturas de la tienda se infieren a partir de los del año 2024, donde la columna _¿Se debería considerar festivo?_ indica si el día debería ser festivo en base a los datos del año 2024, y el campo _¿Se debería haber abierto?_ denota si la tienda debería haber abierto según la información disponible del período de 2024.

In [ ]:
# Resumen de días festivos en el año 2025 de la empresa
festivos_2025: pd.DataFrame = pd.DataFrame({
    "Días": [
        "2025-01-01", "2025-01-06", "2025-04-18", "2025-04-21", "2025-05-01", "2025-05-31", "2025-06-09", "2025-06-24",
        "2025-08-15", "2025-09-11", "2025-10-12", "2025-11-01", "2025-12-06", "2025-12-08", "2025-12-25", "2025-12-26"
    ],
    "Festividades": [
        "Año nuevo", "Día de reyes", "Viernes santo", "Lunes de Pascua", "Día del trabajador", "Día propio de la empresa",
        "Segunda Pascua", "San Juan", "Día de la Asunción de la Virgen María", "Diada de Cataluña", "Fiesta nacional de España",
        "Todos los Santos", "Día de la constitución española", "Día de la Inmaculada Concepción", "Navidad", "San Esteban"
    ],
    "¿Se considera festivo?": [1, 1, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
    "¿Se debería considerar festivo?": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1],
    "¿Se abrió?": [0, 0, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
    "¿Se debería haber abierto?": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
})
festivos_2025["Días"] = pd.to_datetime(festivos_2025["Días"])

festivos_2025.head(festivos_2025.shape[0])

Finalmente, se repite el mismo proceso para las festividades del año 2026, las cuales incluyen únicamente las del mes de enero dado que los períodos de la demanda terminan en marzo de ese mismo año.

In [ ]:
# Resumen de días festivos en el año 2026 de la empresa
festivos_2026: pd.DataFrame = pd.DataFrame({
    "Días": ["2026-01-01", "2026-01-06"],
    "Festividades": ["Año nuevo", "Día de reyes"],
    "¿Se considera festivo?": [np.nan, np.nan],
    "¿Se debería considerar festivo?": [1, 1],
    "¿Se abrió?": [np.nan, np.nan],
    "¿Se debería haber abierto?": [0, 0]
})
festivos_2026["Días"] = pd.to_datetime(festivos_2026["Días"])

festivos_2026.head(festivos_2026.shape[0])

<br><br>
Entonces, con los datos del año 2025 y 2026, se reconstruyen las variables _bolHoliday_ y _bolOpen_. Además, debe tenerse en cuenta que la empresa cierra todos los domingos según los datos del año 2024, exceptuando el último domingo de junio.

In [ ]:
dataset_filtrado["bolOpen_reconstruido"] = dataset_filtrado["bolOpen"]
dataset_filtrado["bolHoliday_reconstruido"] = dataset_filtrado["bolHoliday"]

# Se añaden los datos de las festividades del año 2025
dataset_reconstruido: pd.DataFrame = dataset_filtrado.merge(
    festivos_2025[[
        "Días",
        "¿Se debería considerar festivo?",
        "¿Se debería haber abierto?"
    ]],
    left_on="fecha",
    right_on="Días",
    how="left"
)

# Se añade la nueva información a bolHoliday y bolOpen
dataset_reconstruido.loc[
    dataset_reconstruido["¿Se debería considerar festivo?"].notna(),
    "bolHoliday_reconstruido"
] = (
    dataset_reconstruido.loc[
        dataset_reconstruido["¿Se debería considerar festivo?"].notna(),
        "¿Se debería considerar festivo?"
    ]
)

dataset_reconstruido.loc[
    dataset_reconstruido["¿Se debería haber abierto?"].notna(),
    "bolOpen_reconstruido"
] = (
    dataset_reconstruido.loc[
        dataset_reconstruido["¿Se debería haber abierto?"].notna(),
        "¿Se debería haber abierto?"
    ]
)

# Se eliminan los campos del DataFrame festivos_2025
dataset_reconstruido.drop(columns=[
    "Días",
    "¿Se debería considerar festivo?",
    "¿Se debería haber abierto?"
], inplace=True)

In [ ]:
# Se repite el proceso para las festividades del año 2026
dataset_reconstruido: pd.DataFrame = dataset_reconstruido.merge(
    festivos_2026[[
        "Días",
        "¿Se debería considerar festivo?",
        "¿Se debería haber abierto?"
    ]],
    left_on="fecha",
    right_on="Días",
    how="left"
)

dataset_reconstruido.loc[
    dataset_reconstruido["¿Se debería considerar festivo?"].notna(),
    "bolHoliday_reconstruido"
] = (
    dataset_reconstruido.loc[
        dataset_reconstruido["¿Se debería considerar festivo?"].notna(),
        "¿Se debería considerar festivo?"
    ]
)

dataset_reconstruido.loc[
    dataset_reconstruido["¿Se debería haber abierto?"].notna(),
    "bolOpen_reconstruido"
] = (
    dataset_reconstruido.loc[
        dataset_reconstruido["¿Se debería haber abierto?"].notna(),
        "¿Se debería haber abierto?"
    ]
)

# Se eliminan los campos del DataFrame festivos_2026
dataset_reconstruido.drop(columns=[
    "Días",
    "¿Se debería considerar festivo?",
    "¿Se debería haber abierto?"
], inplace=True)

In [ ]:
# Se marcan como festivos todos los domingos del dataset excepto los de la última semana de junio
dataset_reconstruido.loc[dataset_reconstruido["dia_semana_str"] == "Domingo", "bolHoliday_reconstruido"] = 1
dataset_reconstruido.loc[dataset_reconstruido["dia_semana_str"] == "Domingo", "bolOpen_reconstruido"] = 0
dataset_reconstruido.loc[
    (dataset_reconstruido["fecha"] == "2025-06-29") | (dataset_reconstruido["fecha"] == "2024-06-30"),
    "bolOpen_reconstruido"
] = 1

In [ ]:
# Calendario de festividades y aperturas de la tienda con bolOpen y bolHoliday reconstruidas
calendario_aperturas: pd.DataFrame = dataset_reconstruido.groupby("idSecuencia").agg({
    "año_mes": "first",
    "dia": "first",
    "bolOpen_reconstruido": "first",
    "dia_semana_str": "first",
    "bolHoliday_reconstruido": "first"
})

datos: pd.DataFrame = calendario_aperturas.pivot(
    index="año_mes",
    columns="dia",
    values="bolOpen_reconstruido"
).fillna(0)

fig, ax = plt.subplots(figsize=(10, 8))
cmap: ListedColormap = ListedColormap(["white", "black"])

im = ax.imshow(datos.values, cmap=cmap, aspect="auto")

ax.set_xticks(np.arange(datos.shape[1]))
ax.set_yticks(np.arange(datos.shape[0]))

ax.set_xticklabels(datos.columns)
ax.set_yticklabels(datos.index)

ax.set_xlabel("Día del mes")
ax.set_ylabel("Mes")
ax.set_title("Calendario de festivos y aperturas con bolHoliday y bolOpen reconstruidas")

ax.set_xticks(np.arange(-.5, datos.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-.5, datos.shape[0], 1), minor=True)

# Se marcan los domingos en rojo
domingos: pd.DataFrame = calendario_aperturas[calendario_aperturas["dia_semana_str"] == "Domingo"]

for _, row in domingos.iterrows():
    fila: int = list(datos.index).index(row["año_mes"])
    col: int = list(datos.columns).index(row["dia"])

    ax.add_patch(plt.Rectangle(
        (col - 0.5, fila - 0.5),
        1, 1,
        fill=False,
        edgecolor="red",
        linewidth=2
    ))

# Se añaden las festividades con un cuadrado azul 
festividades: pd.DataFrame = calendario_aperturas[calendario_aperturas["bolHoliday_reconstruido"] == 1]

for _, row in festividades.iterrows():
    fila: int = list(datos.index).index(row["año_mes"])
    col: int = list(datos.columns).index(row["dia"])

    ax.text(
        col, fila,
        "■",
        ha="center",
        va="center",
        color="blue",
        fontsize=10
    )

plt.tight_layout()
plt.show()

El calendario ajustado refleja todas las festividades que la empresa ha celebrado durante el año del que se dispone información, así como los días que no se abrió la tienda. Este análisis extenso permite poder utilizar las variable _bolHoliday_reconstruido_ y _bolOpen_reconstruido_ en los modelos manteniendo la coherencia de los datos ya que, de no haber hecho nada, estos dos atributos hubieran tenido un impacto negativo en la capacidad predictiva de los modelos de aprendizaje automático. Nautralmente esto no es deseable, pero en este caso especialmente ya que estos dos campos aportan un contexto muy rico y amplio para entender la variación de la demanda a lo largo del año.

<a id='ej3.1.6'></a>
### 3.1.6. _diasEntrePedidos_

Esta variable está directamente relacionada con el ciclo de reposición de un producto, que es el tiempo que debe pasar para revisar el inventario de un producto concreto y decidir si se debe realizar un pedido. El valor de este atributo depende de varios factores, como la demanda del propio producto, los tiempos de entrega de los proveedores y la importancia del artículo en cuanto a las ventas de la empresa.

In [ ]:
# DataFrame con la variable diasEntrePedidos por producto
dias_pedidos_analisis: pd.DataFrame = dataset_reconstruido.groupby("producto", as_index=False).agg({
    "diasEntrePedidos": "first"
})

In [ ]:
# diasEntrePedidos para cada producto
plt.figure(figsize=(11, 6))

plt.bar(
    dias_pedidos_analisis["producto"],
    dias_pedidos_analisis["diasEntrePedidos"],
    color="blue",
    alpha=0.6
)

plt.title("diasEntrePedidos para cada producto")
plt.xlabel("producto")
plt.ylabel("diasEntrePedidos")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Histograma de la variable diasEntrePedidos
ax: matplotlib.axes.Axes = dias_pedidos_analisis["diasEntrePedidos"].hist(
    log=True,
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

plt.axvline(
    dias_pedidos_analisis["diasEntrePedidos"].mean(),
    color="blue",
    linestyle="--",
    linewidth=2
)

ax.set_title("Histograma de diasEntrePedidos en escala logarítmica")
ax.set_xlabel("diasEntrePedidos")
ax.set_ylabel("log(Frecuencia)")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot del campo diasEntrePedidos
ax: matplotlib.axes.Axes = dias_pedidos_analisis.boxplot(
    "diasEntrePedidos",
    color="blue"
)

plt.axhline(
    dias_pedidos_analisis["diasEntrePedidos"].mean(),
    color="blue",
    linestyle="--",
    linewidth=2
)

ax.set_title("Boxplot de diasEntrePedidos")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Los dos productos con mayor diasEntrePedidos
print(
    f"Los dos productos con el ciclo de reposición más largo son los de ID {
        ", ".join([str(n) for n in dias_pedidos_analisis.sort_values(by="diasEntrePedidos", ascending=False)["producto"].tolist()[:2]])
    }"
)

El histograma de la variable _diasEntrePedidos_, representado en escala logarítmica, muestra una distribución claramente asimétrica y con una elevada concentración de valores en rangos intermedios. La mayor parte de los productos presentan ciclos de reposición comprendidos aproximadamente entre los 8 y 15 días, lo que sugiere una política de aprovisionamiento relativamente homogénea para la mayoría de referencias. Asimismo, se observa la presencia de un grupo reducido de productos con valores significativamente más elevados, como el 418 y el 868, situados alrededor de los 35 días. El _boxplot_ de la variable permite apreciar que estos casos son poco frecuentes, actuando como valores extremos, los cuales no deben eliminarse ya que estos valores no son fruto de un error en los datos. Por otro lado, el comportamiento observado sugiere la coexistencia de al menos dos tipologías de productos, los que tienen una rotación más frecuente, que requieren reposiciones periódicas en intervalos cortos, y los artículos de menor demanda o mayor estabilidad, cuyo ciclo de aprovisionamiento es más amplio.

<a id='ej3.1.7'></a>
### 3.1.7. _diasLeadtime_

Esta variable resulta muy importante para planificar las compras de productos a proveedores, ya que durante este tiempo la demanda del producto en cuestión debe quedar totalmente cubierta para evitar _stockouts_. Así pues, dado que la naturaleza de este atributo es similar a la de _diasEntrePedidos_, el análisis que se lleva a cabo es bastante similar.

In [ ]:
# DataFrame con el campo diasLeadtime por producto
lead_time_analisis: pd.DataFrame = dataset_reconstruido.groupby("producto", as_index=False).agg({
    "diasLeadtime": "first"
})

In [ ]:
# diasLeadtime para cada producto
plt.figure(figsize=(11, 6))

plt.bar(
    lead_time_analisis["producto"],
    lead_time_analisis["diasLeadtime"],
    color="blue",
    alpha=0.6
)

plt.title("diasLeadtime para cada producto")
plt.xlabel("producto")
plt.ylabel("diasLeadtime")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Histograma de la variable diasLeadtime
ax: matplotlib.axes.Axes = lead_time_analisis["diasLeadtime"].hist(
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

plt.axvline(
    lead_time_analisis["diasLeadtime"].mean(),
    color="blue",
    linestyle="--",
    linewidth=2
)

ax.set_title("Histograma de diasLeadtime")
ax.set_xlabel("diasLeadtime")
ax.set_ylabel("Frecuencia")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot del campo diasEntrePedidos
ax: matplotlib.axes.Axes = lead_time_analisis.boxplot(
    "diasLeadtime",
    color="blue"
)

plt.axhline(
    lead_time_analisis["diasLeadtime"].mean(),
    color="blue",
    linestyle="--",
    linewidth=2
)

ax.set_title("Boxplot de diasLeadtime")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Los dos productos con mayor diasEntrePedidos
print(
    f"Los dos productos con el tiempo de entrega más largo son los de ID {
        ", ".join([str(n) for n in lead_time_analisis.sort_values(by="diasLeadtime", ascending=False)["producto"].tolist()[:2]])
    }"
)

La distribución de la variable _diasLeadtime_ es claramente multimodal, donde se distinguen varios grupos de productos con tiempos de entrega diferenciados. En particular, la mayor parte de las referencias presentan tiempos de entrega reducidos, concentrados aproximadamente entre 1 y 5 días, lo que sugiere una cadena de suministro ágil para una parte significativa del catálogo, por lo que pueden tratarse de ítems como ambientadores o artículos de decoración del vehículo. Por otra banda, también se identifica un segundo grupo relevante de productos con tiempos de entrega entorno a los 14–15 días, lo que podría estar asociado a artículos que requieren aprovisionamiento desde ubicaciones más alejadas o proveedores con menor frecuencia de suministro. Finalmente, se observa la presencia de un número muy reducido de productos con tiempos de entrega aún mayores, cercanos a los 20 días, que pueden considerarse casos extremos dentro de la distribución, los cuales podrían ser piezas del automóvil que pueden provenir de otros países, hecho que aumenta la complejidad logística del aprovisionamiento de este tipo de artículos.

Para complementar el análisis de _diasEntrePedidos_ y _diasLeadtime_, resulta interesante analizar de forma breve la variable _diasAprovisionamiento_, que es la suma de los otros dos campos mencionados. Este intervalo de tiempo se denomina intervalo de protección, y es el período de tiempo durante el cual se debe satisfacer la demanda sin poder beneficiarse de nuevas decisiones de reposición.

In [ ]:
# DataFrame con el ciclo de aprovisionamiento completo
ciclo_aprovisionamiento_analisis: pd.DataFrame = dias_pedidos_analisis.merge(
    lead_time_analisis,
    how="left",
    left_on="producto",
    right_on="producto"
)

ciclo_aprovisionamiento_analisis["diasAprovisionamiento"] = (
    ciclo_aprovisionamiento_analisis["diasEntrePedidos"]
    + ciclo_aprovisionamiento_analisis["diasLeadtime"]
)

In [ ]:
# Histograma de la variable diasAprovisionamiento
ax: matplotlib.axes.Axes = ciclo_aprovisionamiento_analisis["diasAprovisionamiento"].hist(
    log=True,
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

ax.set_title("Histograma de diasAprovisionamiento en escala logarítmica")
ax.set_xlabel("diasAprovisionamiento")
ax.set_ylabel("log(Frecuencia)")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot del atributo diasAprovisionamiento
ax: matplotlib.axes.Axes = ciclo_aprovisionamiento_analisis.boxplot(
    "diasAprovisionamiento",
    color="blue"
)

plt.axhline(
    ciclo_aprovisionamiento_analisis["diasAprovisionamiento"].mean(),
    color="blue",
    linestyle="--",
    linewidth=2
)

ax.set_title("Boxplot de diasAprovisionamiento")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

El histograma logarítmico de _diasAprovisionamiento_ refleja una distribución dispersa y multimodal del atributo. La mayor concentración de productos se sitúa en torno a valores intermedios, aproximadamente entre 12 y 20 días, lo que indica horizontes de aprovisionamiento relativamente cortos para la mayoría de referencias. No obstante, se identifican también grupos diferenciados con valores más elevados, entorno a los 30 días y superiores, así como algunos casos extremos que superan los 50 días. Estos valores corresponden a productos con ciclos de reposición más largos y/o mayores tiempos de entrega, lo que implica una mayor exposición al riesgo de rotura de _stock_.

<a id='ej3.1.8'></a>
### 3.1.8. _eurPrecioMedio_

El precio promedio de cada producto es una variable numérica que influye directamente en la cantidad de productos que los consumidores están dispuestos a comprar. Tal es la importancia de esta variable que la previsión de la demanda basada en el precio facilita la implementación de estrategias de precios dinámicos que se ajustan en tiempo real a las condiciones del mercado, las preferencias de los consumidores y los ingresos disponibles, permitiendo a la compañía maximizar sus beneficios.

In [ ]:
# DataFrame con el atributo eurPrecioMedio para cada artículo
precio_analisis: pd.DataFrame = dataset_reconstruido.groupby("producto", as_index=False).agg({
    "eurPrecioMedio": "first"
})

In [ ]:
# eurPrecioMedio por ítem
plt.figure(figsize=(12, 6))

plt.bar(
    precio_analisis["producto"],
    precio_analisis["eurPrecioMedio"],
    color="blue",
    alpha=0.6
)

plt.title("eurPrecioMedio por artículo")
plt.xlabel("producto")
plt.ylabel("eurPrecioMedio (€)")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Histograma de eurPrecioMedio
ax: matplotlib.axes.Axes = precio_analisis["eurPrecioMedio"].hist(
    log=True,
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

ax.set_title("Histograma de eurPrecioMedio en escala logarítmica")
ax.set_xlabel("eurPrecioMedio")
ax.set_ylabel("log(Frecuencia)")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot del campo eurPrecioMedio
ax: matplotlib.axes.Axes = precio_analisis.boxplot(
    "eurPrecioMedio",
    color="blue"
)

plt.axhline(
    precio_analisis["eurPrecioMedio"].mean(),
    color="blue",
    linestyle="--",
    linewidth=2
)

ax.set_title("Boxplot de eurPrecioMedio")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Los cinco ítems de mayor precio promedio
print(
    f"Los productos con el precio medio de venta más alto son los de ID {
        ", ".join([str(n) for n in precio_analisis.sort_values(by="eurPrecioMedio", ascending=False)["producto"].tolist()[:5]])
    }"
)

In [ ]:
# Precio promedio frente a la demanda media
precio_demanda_analisis: pd.DataFrame = dataset.groupby("producto").agg(
    precio=("eurPrecioMedio", "first"),
    demanda_media=("udsVenta", "mean")
)

plt.scatter(
    precio_demanda_analisis["precio"],
    precio_demanda_analisis["demanda_media"],
    color="blue",
    alpha=0.5
)

plt.title("Relación entre precio y demanda promedio")
plt.xlabel("eurPrecioMedio (€)")
plt.ylabel("Promedio de udsVenta")

plt.tight_layout()
plt.show()

La distribución logarítmica de la variable _eurPrecioMedio_ muestra una concentración de productos en rangos de precio bajos y una larga cola hacia valores altos. Este comportamiento se confirma en el _boxplot_, donde se observa una mediana reducida y la presencia de numerosos valores atípicos asociados a productos de mayor precio. Este patrón refleja un catálogo heterogéneo, en el que predominan productos económicos junto con un número reducido de referencias de alto valor. Este hecho se ve reforzado al analizar la relación entre el precio y la demanda promedio, donde se observa una tendencia inversa donde los productos más baratos presentan, en general, mayores niveles de demanda, mientras que los más caros tienden a concentrarse en valores más bajos. Esto puede sugerir que los productos más costosos pueden ser de baja urgencia en el contexto de la empresa, siendo productos del automóvil que únicamente son necesarios en casos extremos como averías o accidentes.

<a id='ej3.2'></a>
## 3.2. Análisis de la variable objetivo

La demanda histórica de cada producto es la variable que se desea predecir a futuro, cuyo impacto en la gestión de un almacén es de suma importancia para optimizar los inventarios y reducir los costes. Por lo tanto, se dedica un apartado a su análisis donde, además de analizar la distribución de este atributo, también se aplica la prueba de _Dickey-Fuller_ para comprobar si las propiedades estadísticas de esta variable se mantienen constantes a lo largo del tiempo, lo cual se presupone en algunos modelos predictivos que se aplicarán más adelante.

In [ ]:
# Histograma de udsVenta
ax: matplotlib.axes.Axes = dataset_reconstruido["udsVenta"].hist(
    log=True,
    bins=20,
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

ax.set_title("Histograma de udsVenta")
ax.set_xlabel("udsVenta")
ax.set_ylabel("Frecuencia")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot de la variable objetivo
ax: matplotlib.axes.Axes = dataset_reconstruido.boxplot(
    "udsVenta",
    color="blue"
)

plt.axhline(
    dataset_reconstruido["udsVenta"].mean(),
    color="blue",
    linestyle="--",
    linewidth=2
)

ax.set_title("Boxplot de udsVenta")

ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Proporción de días con 0 ventas por producto
cero_ventas: int = dataset_reconstruido[dataset_reconstruido["udsVenta"] == 0].shape[0]
total: int = dataset_reconstruido.shape[0]
prop_cero_ventas: float = round(cero_ventas / total, 2)

plt.bar(
    ["Cero ventas", "Al menos una venta"],
    [prop_cero_ventas, 1 - prop_cero_ventas],
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

plt.title("Proporciones de ventas")
plt.ylabel("Proporción (%)")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Los dos productos que registraron más ventas en un solo día
productos: List[int] = dataset_reconstruido.sort_values(by="udsVenta", ascending=False)["producto"].tolist()[:2]
ventas: List[int] = dataset_reconstruido.sort_values(by="udsVenta", ascending=False)["udsVenta"].tolist()[:2]

print(
    f"Los productos con ID {", ".join([str(n) for n in productos])} obtuvieron un total de {", ".join(str(n) for n in ventas)} "
    "ventas en un solo día, respectivamente"
)

La empresa acostumbra a registrar ventas de menos de 50 unidades por producto en la gran mayoría de artículos, donde en la mayoría de casos no se produce ninguna transacción. Por otro lado, existe un pequeño subconjunto de ventas que se encuentran entre las 50 y las 200 unidades en un solo día, lo cual puede deberse a casos excepcionales donde productos de alta necesidad fueron sujetos a un promoción en un momento muy concreto donde la demanda era muy alta. Por último, existen dos días en que se registraron 397 y 187 ventas de un solo producto, siendo estos los de ID 669 y 228, lo cual son casos excepcionales que han ocurrido únicamente dos veces en dos años.

Teniendo esto en cuenta, con el objetivo de ilustrar la demanda de un conjunto de ítems del _dataset_, se representa dicha variable de 4 artículos de los que se han observado características interesantes a lo largo del análisis exploratorio. Estos son el 669, que presenta el día con el mayor número de ventas de todo el conjunto de datos, el 572, que tiene el precio de venta más alto de todo el juego de datos, el 544, cuyas ventas son las que más mejoran cuando se incluye en una promoción, y el 904, cuyas ventas se desploman considerablemente cuando se pone en oferta.

In [ ]:
# Se grafica la demanda de los 4 productos mencionados
productos: List[int] = [669, 572, 544, 904]

fig, axs = plt.subplots(nrows=len(productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, productos):
    dataframe_producto: pd.DataFrame = dataset_reconstruido[dataset_reconstruido["producto"] == producto]

    ax.plot(
        dataframe_producto["fecha"],
        dataframe_producto["udsVenta"],
        color="blue"
    )

    ax.set_title(f"udsVentas del producto {producto}")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Unidades")
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    
plt.tight_layout()
plt.show()

Las series temporales representadas permiten inferir posibles diferencias en la naturaleza de los productos. En el caso del ítem con mayor número de ventas en un solo día, el 669, se observa una demanda altamente intermitente con picos muy pronunciados, lo que sugiere que podría tratarse de un artículo de alta rotación sujeto a eventos puntuales o campañas específicas, como consumibles o productos ampliamente demandados en determinadas situaciones. Por el contrario, el producto con mayor precio, el 572 presenta una demanda más contenida y relativamente estable, con valores bajos y sin picos extremos, lo que resulta coherente con artículos de mayor coste asociados a necesidades específicas o menos frecuentes, como componentes más técnicos o de sustitución. Esto mismo también ocurre con los otros dos productos, el 544 y el 904, ya que también se observa un patrón de demanda constante a lo largo de toda la serie. 

Por otro lado, con el objetivo de analizar la estacionariedad de algunas de las series temporales del _dataset_, se aplica el test de _Dickey-Fuller_ aumentado al conjunto representativo de productos de la empresa empleado anteriormente, es decir, los de ID 669, 572, 544, y 904. En este caso, se escoge una serie de artículos cuyas series temporales 

In [ ]:
# Función que aplica el test de Dickey-Fuller a una serie temporal
def test_adf(serie: pd.Series, producto: int, nivel_significancia=0.05) -> None:
    """Aplica la prueba de Dickey-Fuller a una serie de la demanda histórica de un producto y muestra
    los resultados.

    Argumentos:
        serie: pd.Series -> Serie temporal de un producto concreto
        producto: int -> ID del producto de la serie temporal
        nivel_significancia: float -> Umbral de probabilidad para decidir si el resultado es estadísticamente
        significativo

    Devuelve:
        None
    """
    resultado: Tuple[Any] = adfuller(serie)
    
    print(f"Estadístico ADF: {resultado[0]:.4f}")
    print(f"p-valor: {resultado[1]:.4f}")
    
    if resultado[1] < nivel_significancia:
        print(f"La serie del producto {producto} sí es estacionaria")
    else:
        print(f"La serie del producto {producto} no es estacionaria")

In [ ]:
# Estacionariedad de la demanda de los productos 669, 572, 544 y 904
for producto in productos:
    serie: pd.Series = dataset_reconstruido[dataset_reconstruido["producto"] == producto]["udsVenta"]
    test_adf(serie, producto)
    print("\n")

La aplicación del test de _Dickey-Fuller_ a las series temporales de los productos analizados demuestra que estas pueden considerarse estacionarias. Este resultado es coherente con la apariencia de los gráficos, donde no se observa una tendencia clara a largo plazo ni cambios estructurales en el nivel medio de la serie. En particular, el producto 669, a pesar de presentar un comportamiento altamente intermitente con numerosos días sin ventas y picos puntuales, mantiene una media y variabilidad relativamente constantes en el tiempo. Por su parte, los productos 572, 544 y 904, muestran una evolución más estable y con menor dispersión, reforzando visualmente la ausencia de tendencia.

A modo de ejemplificar un caso donde la serie temporal de la demanda no es estacionaria, se analiza el caso del producto número 7. El gráfico de la demanda de este artículo permite intuir esta propiedad en este caso, ya que a mediados del año 2025 se observa una demanda alta y constante con apenas días sin ventas, hecho que no ocurrió en la misma época del año 2024. Este comportamiento no estacionario se ve reforzado con la aplicación de la prueba _Dickey-Fuller_, donde el _p-valor_ de 0.5 unidades es mayor que el nivel de significancia establecido y, por lo tanto, se acepta la hipótesis nula que plantea que la serie temporal no es estacionaria. 

In [ ]:
# Ejemplo de serie temporal no estacionaria
producto: int = 7
plt.figure(figsize=(10, 5))

dataframe_producto: pd.DataFrame = dataset_reconstruido[dataset_reconstruido["producto"] == producto]
plt.plot(dataframe_producto["fecha"], dataframe_producto["udsVenta"], color="blue")

plt.title(f"udsVentas del producto {producto}")
plt.xlabel("Fecha")
plt.ylabel("Unidades")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)
    
plt.tight_layout()
plt.show()

In [ ]:
# Test Dickey-Fuller aplicado a una serie no estacionaria
serie: pd.Series = dataset_reconstruido[dataset_reconstruido["producto"] == producto]["udsVenta"]
test_adf(serie, producto)

A modo de conclusión en cuanto a la estacionariedad de las series temporales, se observa que algunos productos sí que describen un comportamiento de este tipo mientras que algunos otros no. Esto es especialmente relevante en cuanto a la aplicación de modelos predictivos estadísticos, los cuales asumen que los datos son estacionarios. No obstante, algunos de estos métodos permiten transformar internamente las series temporales para que cumplan esta condición mediante la diferenciación de puntos continuos. Esto se lleva a cabo de forma automática, ya que los propios modelos se encargan de ajustar el grado de diferenciación necesario, por lo que no resulta imprescindible aplicar transformaciones manuales a todas las series.

<a id='ej3.3'></a>
## 3.3. Análisis conjunto

Una vez se han analizado todas las variables que se introducirán en los _datasets_ que se usarán como _input_ en los modelos predictivos, se lleva a cabo un breve análisis conjunto donde se examinan las correlaciones entre todos los atributos del juego de datos.

In [ ]:
corr: pd.DataFrame = dataset_reconstruido[[
    "udsVenta",
    "isPromo",
    "bolHoliday_reconstruido",
    "bolOpen_reconstruido",
    "diasEntrePedidos",
    "diasLeadtime",
    "eurPrecioMedio",
    "dia_semana_int",
    "mes_int"
]].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Matriz de correlaciones")

plt.tight_layout()
plt.show()

La matriz de correlación muestra que ninguna de las variables explicativas presenta una relación lineal fuerte con la variable objetivo, observándose en todos los casos coeficientes de baja magnitud. En particular, variables como las promociones o el precio medio presentan correlaciones prácticamente nulas, mientras que otras como el _lead time_ o la apertura del establecimiento muestran relaciones ligeramente positivas, aunque débiles. Este resultado sugiere que la demanda no depende de forma lineal de ninguna de las variables consideradas de manera aislada, sino que su comportamiento está influido por múltiples factores y, posiblemente, por interacciones entre ellos. Asimismo, cabe destacar la fuerte correlación negativa entre _bolHoliday_reconstruido_ y _bolOpen_reconstruido_, lo cual es coherente desde un punto de vista operativo, ya que los días festivos suelen coincidir con el cierre del establecimiento. Por otro lado, _dia_semana_int_ tiene una correlación positiva con _bolHoliday_reconstruido_, hecho que se debe a que la mayoría de días festivos en el _dataset_ caen en días a final de la semana. Sin embargo, ocurre lo contrario entre los días de la semana y _bolOpen_reconstruido_, dado que la mayoría de días que la tienda abre se encuentran al inicio de este período.

Por otro lado, esta ausencia de de correlaciones lineales fuertes observada no implica que las variables carezcan de capacidad explicativa, ya que su efecto puede depender del tipo de producto considerado. Esto sucede porque estos valores se han calculado sobre el conjunto completo de productos, lo que puede ocultar relaciones existentes a nivel individual. Dado que el _dataset_ está compuesto por múltiples series temporales de distintos artículos, es posible que determinadas variables presenten una relación significativa con la demanda en algunos ítems concretos, mientras que en otros dicha relación sea inexistente o incluso opuesta. Este fenómeno puede diluir las correlaciones globales, dando lugar a coeficientes reducidos que no reflejan adecuadamente el comportamiento real de la demanda a nivel individual. Este hecho plantea la posibilidad de segmentar los productos en grupos homogéneos, de modo que las relaciones entre variables puedan analizarse de forma más precisa y ser mejor aprovechadas en los modelos de predicción.

Dicho esto, se procede a visualizar los datos a nivel de producto mediante técnicas de reducción de la dimensionalidad como _PCA_, que permite capturar relaciones lineales entre las variables, y _t-SNE_, que también puede capturar relaciones no lineales. El objetivo es analizar la estructura interna de los productos en función de sus características agregadas. Estas características son atributos que o bien se han creado en algún momento del análisis exploratorio o bien son variables como _eurPrecioMedio_ o _diasEntrePedidos_ que se expresan a nivel de artículo. A continuación se describen brevemente los distintos campos agregados.

<ul>
    <li>Variables a nivel de producto &#8594; <i>diasEntrePedidos</i>, <i>diasLeadtime</i>, <i>eurPrecioMedio</i></li>
    <li><i>promedio_ventas</i> &#8594; Media aritmética de <i>udsVenta</i> por producto</li>
    <li><i>std_ventas</i> &#8594; Desviación estándar de <i>udsVenta</i> por producto</li>
    <li><i>pct_cero_ventas</i> &#8594; Porcentaje de cero ventas en el histórico de la demanda por producto</li>
    <li><i>pct_promo</i> &#8594; Porcentaje de días en los que el producto formó parte de una oferta</li>
    <li><i>mejora_promo</i> &#8594; Diferencia entre el promedio de ventas cuando el producto forma parte de una promoción y cuando no está en oferta</li>    
</ul>

In [ ]:
# DataFrame con información distintiva de cada producto
subset = dataset_reconstruido.groupby("producto", as_index=False).agg(
    promedio_ventas=("udsVenta", "mean"),
    pct_cero_ventas=("udsVenta", lambda g: (g == 0).mean()),
    std_ventas=("udsVenta", "std"),
    pct_promo=("isPromo", "mean"),
    promedio_ventas_promo=("udsVenta", lambda g: g[dataset_reconstruido.loc[g.index, "isPromo"] == 1].mean()),
    promedio_ventas_no_promo=("udsVenta", lambda g: g[dataset_reconstruido.loc[g.index, "isPromo"] == 0].mean()),
    diasEntrePedidos=("diasEntrePedidos", "first"),
    diasLeadtime=("diasLeadtime", "first"),
    eurPrecioMedio=("eurPrecioMedio", "first")
)

subset["mejora_promo"] = subset["promedio_ventas_promo"] - subset["promedio_ventas_no_promo"]
subset["mejora_promo"] = subset["mejora_promo"].fillna(0)
subset.drop(["promedio_ventas_promo", "promedio_ventas_no_promo"], axis=1, inplace=True)

In [ ]:
# Estandarización de las variables
X: pd.DataFrame = subset.drop("producto", axis=1)

scaler: StandardScaler = StandardScaler()
X_escalado: np.ndarray = scaler.fit_transform(X)

In [ ]:
# PCA y gráfico en dos dimensiones
pca: PCA = PCA(n_components=2)
pca_x: np.ndarray = pca.fit_transform(X_escalado)

plt.figure(figsize=(8, 6))

sns.scatterplot(
    x=pca_x[:, 0],
    y=pca_x[:, 1],
    hue=subset["producto"],
    alpha=0.7
)

plt.title("Reducción 2D mediante PCA")
plt.xlabel("CP 1")
plt.ylabel("CP 2")

plt.grid(True, alpha=0.3)
plt.legend("", frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# t-SNE y gráfico en dos dimensiones
SEMILLA: int = 42

tsne: TSNE = TSNE(n_components=2, perplexity=50, learning_rate="auto", max_iter=1000, random_state=SEMILLA)
tsne_x = tsne.fit_transform(X_escalado)

plt.figure(figsize=(8, 6))

sns.scatterplot(
    x=tsne_x[:,0],
    y=tsne_x[:,1],
    hue=subset["producto"],
    alpha=0.7
)

plt.title("Reducción 2D mediante t-SNE")
plt.xlabel("TSNE 1")
plt.ylabel("TSNE 2")

plt.grid(True, alpha=0.3)
plt.legend("", frameon=False)

plt.tight_layout()
plt.show()

En el caso de _PCA_, se observa una distribución relativamente continua de los datos, sin una separación clara entre grupos, lo que sugiere que las diferencias entre productos no se explican únicamente mediante combinaciones lineales de las variables consideradas. Por otro lado, la proyección mediante _t-SNE_ revela la existencia de agrupaciones más definidas, con varios núcleos de puntos claramente diferenciados en el espacio reducido. Esto indica que, aunque las relaciones entre variables no son lineales, como ya se observó en la matriz de correlación, sí existen patrones estructurales en los datos que permiten distinguir distintos tipos de productos en función de su comportamiento.

A modo de conclusión, estos resultados refuerzan la hipótesis de que los productos presentan una heterogeneidad significativa y que es posible agruparlos en grupos homogéneos, lo que justifica la aplicación de técnicas de _clustering_.

<br><br><a id='ej4'></a>
## 4. _Clustering_

Durante los distintos apartados de este _notebook_, se ha demostrado que los productos presentan características heterogéneas, ya que algunos presentan tendencias estacionarias y otros que no, algunos tienen picos de demanda muy esporádicos, y otros presentan series temporales muy volátiles. Estas diferencias entre los patrones de la demanda provocan que todos los artículos difícilmente pueden pronosticarse de la misma manera. Esto resulta especialmente relevante para el entrenamiento de los modelos de aprendizaje automático, donde se concibe la posibilidad de entrenar modelos que incluyan múltiples productos parecidos entre sí. Dicho esto, la heterogeneidad observada en los datos implica la necesidad de agrupar los productos del juego de datos en base a sus similitudes. 

Para esta tarea se emplea el algoritmo _k-means_, que es un modelo no supervisado que se basa en la minimización de la varianza dentro de los grupos mediante el cálculo de distancias y centroides. Una particularidad de este algoritmo es que se debe fijar un número de grupos antes de ejecutarlo, de manera que, para encontrar el número óptimo de _clusters_, se lanza el algoritmo para una serie de valores de grupos y para cada uno se calcula la suma de errores cuadráticos. El resultado de este proceso es el gráfico _Regla del codo del algoritmo k-means_, donde la suma de errores cuadráticos disminuye a medida que se aumenta el número de _clusters_. Consecuentemente, se considera que el valor óptimo de grupos es el que se sitúa en el codo de la visualización, siendo 10 en este caso, ya que valores más grandes implican una reducción del error demasiado baja a cambio de introducir un número de grupos demasiado elevado, dificultando la interpretación de cada _cluster_.

In [ ]:
# Selección del mejor número de clusters para k-means
k_INI: int = 1
k_FIN: int = 30
n_INI: int = 10

sse_k: List[float] = []
for k in range(k_INI, k_FIN):
    # Se inicializa la clase KMeans() y se aplica a los datos
    modelo_kmeans: KMeans = KMeans(n_clusters=k, n_init=n_INI, random_state=SEMILLA)
    modelo_kmeans.fit(X_escalado)

    # Se recogen las etiquetas de los datos y los centroides de sus grupos asignados
    etiquetas: np.ndarray = modelo_kmeans.labels_
    centroides: np.ndarray = modelo_kmeans.cluster_centers_

    # Se empareja cada dato con su centroide
    asignacion: Dict[Tuple[Any], np.ndarray] = {tuple(x): centroides[etiqueta] for x, etiqueta in zip(X_escalado, etiquetas)}

    # Se calcula la suma de errores cuadráticos
    sse: int = 0
    for (punto, centroide), etiqueta in zip(asignacion.items(), etiquetas):
        p: np.ndarray = np.array(punto)
        c: np.ndarray = np.array(centroide)
        sse += np.linalg.norm(p - c)**2
    sse_k.append(round(sse, 2))
    
# Se visualizan los resultados
plt.figure(figsize=(8, 6))
plt.plot(range(k_INI, k_FIN), sse_k, color="blue")
plt.scatter(range(k_INI, k_FIN), sse_k, marker="o", color="blue")

plt.title(f"Regla del codo del algoritmo k-means")
plt.xlabel("Número de clusters")
plt.ylabel("Suma de errores cuadrados")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Aplicación de k-means con el mejor número de clusters y visualización con t-SNE
n_CLUSTERS: int = 10

modelo_kmeans_opt: KMeans = KMeans(n_clusters=n_CLUSTERS, n_init=n_INI, random_state=SEMILLA)
modelo_kmeans_opt.fit(X_escalado)

plt.figure(figsize=(10, 8))

etiquetas: np.ndarray = modelo_kmeans_opt.labels_
scatter: plt.scatter = plt.scatter(tsne_x[:, 0], tsne_x[:, 1], c=etiquetas, cmap="tab20")

plt.title("Clusters con KMeans reducido con t-SNE (k=10)")
plt.xlabel("TSNE 1")
plt.ylabel("TSNE 2")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

valores_unicos: np.ndarray = np.unique(etiquetas)
norm: Normalize = Normalize(vmin=etiquetas.min(), vmax=etiquetas.max())

handles: list[Line2D] = []
labels_leyenda: list[str] = []

for v in valores_unicos:
    color = scatter.cmap(norm(v))
    
    handles.append(
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor=color,
               markeredgecolor=color, markersize=8)
    )
    labels_leyenda.append(f"Cluster {int(v)}")

plt.legend(handles, labels_leyenda, title="Clusters")

plt.tight_layout()
plt.show()

In [ ]:
# Aplicación de k-means con el mejor número de clusters y visualización con PCA
plt.figure(figsize=(10, 8))

etiquetas: np.ndarray = modelo_kmeans_opt.labels_
scatter: plt.scatter = plt.scatter(pca_x[:, 0], pca_x[:, 1], c=etiquetas, cmap="tab20")

plt.title("Clusters con KMeans reducido con PCA (k=10)")
plt.xlabel("CP 1")
plt.ylabel("CP 2")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

valores_unicos: np.ndarray = np.unique(etiquetas)
norm: Normalize = Normalize(vmin=etiquetas.min(), vmax=etiquetas.max())

handles: list[Line2D] = []
labels_leyenda: list[str] = []

for v in valores_unicos:
    color = scatter.cmap(norm(v))
    
    handles.append(
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor=color,
               markeredgecolor=color, markersize=8)
    )
    labels_leyenda.append(f"Cluster {int(v)}")

plt.legend(handles, labels_leyenda, title="Clusters")

plt.tight_layout()
plt.show()

La proyección mediante _PCA_ no presenta una separación clara entre los productos, mientras que el uso de _t-SNE_ muestra la existencia de _clusters_ claramente diferenciados, junto con otros grupos más difusos y algunos conjuntos pequeños de productos con comportamientos atípicos esparcidos por distintas partes del plano bidimensional de _t-SNE_. Este comportamiento sugiere la existencia de relaciones no lineales entre las variables, que no son capturadas por métodos lineales como PCA. Además, esta estructura refuerza la necesidad de segmentar el dataset antes del modelado, permitiendo el entrenamiento de modelos específicos para grupos homogéneos. Dicho esto, para poder analizar qué caracteriza a cada _cluster_, se visualiza la distribución de los tamaños de cada grupo junto con un _heatmap_ que resalta las diferencias relativas entre los promedios de las variables de los grupos frente al _dataset_ completo. Esto permite identificar si existen grupos realmente distintos o, en caso contrario, si algunos _clusters_ presentan características internas parecidas.

In [ ]:
# Tamaño de cada cluster
subset["cluster_kmeans"] = etiquetas
tamaños: List[int] = subset.groupby("cluster_kmeans").size().tolist()

plt.bar(
    range(n_CLUSTERS),
    tamaños,
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

plt.title("Productos por cluster")
plt.xlabel("Cluster")
plt.xticks(range(n_CLUSTERS))
plt.ylabel("Productos")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Análisis intercluster de k-means respecto al promedio de todos los productos
resumen: pd.DataFrame = subset.drop("producto", axis=1).groupby("cluster_kmeans").mean()
resumen_relativo: pd.DataFrame = (
    (resumen - subset.drop(columns=["producto", "cluster_kmeans"]).mean()) 
    / subset.drop(columns=["producto", "cluster_kmeans"]).std()
)

plt.figure(figsize=(8, 8))

sns.heatmap(resumen_relativo, cmap="coolwarm", center=0)

plt.title("Diferencias relativas entre los clusters y el promedio de los productos")

plt.tight_layout()
plt.show()

El mapa de calor representado junto con la reducción mediante _t-SNE_ permiten identificar qué hace a cada grupo único y cómo se refleja esto en el espacio reducido. A continuación se listan las características importantes de cada agrupación.

<ul>
    <li><i>Cluster</i> 0 &#8594; Presenta un tamaño considerable con más de 100 productos con un tiempo de entrega moderado</li>
    <li><i>Cluster</i> 1 &#8594; Contiene una cantidad de artículos similar al grupo anterior pero con una ligera mejora de ventas cuando los productos se promocionan</li>
    <li><i>Cluster</i> 2 &#8594; Es uno de los grupos más pequeños y se diferencia por una volatilidad de ventas muy grande dada por la variable <i>std_ventas</i>, donde muchos días no tienen ventas</li>
    <li><i>Cluster</i> 3 &#8594; Contiene pocos productos que se caracterizan por el gran tiempo en el que están de oferta</li>
    <li><i>Cluster</i> 4 &#8594; Presenta alrededor de 75 artículos con una rotación en el inventario relativamente alta</li>
    <li><i>Cluster</i> 5 &#8594; Es de lejos el <i>cluster</i> más grande con más de 200 productos, donde el comportamiento es muy cercano a la media exceptuando una cantidad de ventas ligeramente por debajo del promedio del <i>dataset</i></li>
    <li><i>Cluster</i> 6 &#8594; Contiene 50 ítems y es el grupo con un promedio de ventas más grande de todos que a su vez presenta muy pocos días sin ninguna venta
    <li><i>Cluster</i> 7 &#8594; Con prácticamente 150 artículos, este grupo es el que presenta un comportamiento más cercano a la media de todos
    <li><i>Cluster</i> 8 &#8594; Este grupo es de los más pequeños y presenta el mayor ciclo de aprovisionamiento de todos, con el ciclo de reposición y los días de entrega más elevados respecto a la media del conjunto de datos
    <li><i>Cluster</i> 9 &#8594; Presenta menos de 50 artículos y se caracteriza por una alta concentración de ítems caros
</ul>

En primer lugar, se observa que los clusters 1, 5 y 7 son los más numerosos y presentan características muy próximas a la media del conjunto de datos. Este comportamiento se refleja también en la proyección bidimensional mediante _t-SNE_, donde dichos grupos conforman la principal aglomeración de puntos situada en la zona inferior izquierda de la representación. Adicionalmente, a pesar de que no se ha incluido en la versión final del presente _notebook_, el análisis de las matrices de correlación de cada uno de estos _clusters_ muestra patrones prácticamente idénticos en la relación entre las variables y la demanda. En consecuencia, se considera que las diferencias entre estos grupos no son suficientemente significativas como para justificar el entrenamiento de modelos independientes, por lo que se opta por su fusión en un único grupo, reduciendo así la complejidad del problema sin pérdida apreciable de información.

Por otro lado, también se identifican regiones bien diferenciadas en el espacio reducido. En particular, el grupo de la parte superior izquierda está compuesto mayoritariamente por productos del _cluster_ 4, mientras que la zona superior derecha corresponde principalmente al _cluster_ 0, lo que indica la existencia de patrones de comportamiento claramente diferenciados respecto al conjunto global. Asimismo, el _cluster_ 6 presenta características similares al _cluster_ 0, aunque con valores más extremos, especialmente en términos de volumen de ventas. Sin embargo, el reducido número de productos que contiene limita su utilidad para el entrenamiento de modelos de aprendizaje automático. Por este motivo, se propone fusionar ambos grupos, no solo por su proximidad en el espacio de características, sino también por la relevancia que presentan en la modelización de la demanda al concentrar productos con un número de ventas elevado.

Finalmente, el resto de _clusters_ presentan un tamaño reducido y, en algunos casos, comportamientos más específicos o atípicos. Dado que no cuentan con suficientes observaciones para entrenar modelos robustos de forma individual, se opta por agruparlos en un _cluster_ residual.

In [ ]:
# Se añaden las etiquetas de los nuevos clusters al subset
subset["cluster_final"] = subset["cluster_kmeans"]
subset.loc[subset["cluster_final"] == 6, "cluster_final"] = 0
subset.loc[subset["cluster_final"].isin([5, 7]), "cluster_final"] = 1
subset.loc[subset["cluster_final"].isin([3, 8, 9]), "cluster_final"] = 2
subset.loc[subset["cluster_final"] == 4, "cluster_final"] = 3

subset["cluster_nombre"] = subset["cluster_final"].astype(str)
subset.loc[subset["cluster_nombre"] == "0", "cluster_nombre"] = "top_ventas"
subset.loc[subset["cluster_nombre"] == "1", "cluster_nombre"] = "estandar"
subset.loc[subset["cluster_nombre"] == "2", "cluster_nombre"] = "residual"
subset.loc[subset["cluster_nombre"] == "3", "cluster_nombre"] = "alta_rotacion"

In [ ]:
# Visualización de los clusters finales con PCA
etiquetas: pd.Categorical = pd.Categorical(subset["cluster_nombre"])
codigos: np.ndarray = etiquetas.codes

cmap: Colormap = plt.cm.get_cmap("tab20", len(etiquetas.categories))

plt.figure(figsize=(10, 8))

plt.scatter(pca_x[:, 0], pca_x[:, 1], c=codigos, cmap=cmap)

plt.title("Clusters finales con KMeans reducido con PCA")
plt.xlabel("CP 1")
plt.ylabel("CP 2")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

handles: List[Line2D] = []
for i, nombre in enumerate(etiquetas.categories):
    color: tuple[float, float, float, float] = cmap(i)
    
    handles.append(
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor=cmap(i), markeredgecolor=cmap(i), label=nombre, markersize=8)
    )

plt.legend(handles=handles, title="Clusters")

plt.tight_layout()
plt.show()

In [ ]:
# Visualización de los clusters finales con t-SNE
etiquetas: pd.Categorical = pd.Categorical(subset["cluster_nombre"])
codigos: np.ndarray = etiquetas.codes

cmap: Colormap = plt.cm.get_cmap("tab20", len(etiquetas.categories))

plt.figure(figsize=(10, 8))

plt.scatter(tsne_x[:, 0], tsne_x[:, 1], c=codigos, cmap=cmap)

plt.title("Clusters finales con KMeans reducido con t-SNE")
plt.xlabel("TSNE 1")
plt.ylabel("TSNE 2")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

handles: List[Line2D] = []
for i, nombre in enumerate(etiquetas.categories):
    color: tuple[float, float, float, float] = cmap(i)
    
    handles.append(
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor=cmap(i), markeredgecolor=cmap(i), label=nombre, markersize=8)
    )

plt.legend(handles=handles, title="Clusters")

plt.tight_layout()
plt.show()

In [ ]:
# Tamaño de cada cluster final
tamaños: List[int] = subset.groupby("cluster_nombre").size().tolist()

plt.bar(
    sorted(subset["cluster_nombre"].unique()),
    tamaños,
    color="blue",
    edgecolor="black",
    linewidth=1,
    alpha=0.6
)

plt.title("Productos por clusters finales")
plt.xlabel("Cluster")
plt.xticks(range(len(subset["cluster_nombre"].unique())))
plt.ylabel("Productos")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

Es importante comentar el hecho que los 4 _clusters_ tienen tamaños dispares, especialmente el grupo _estandar_, que es el que acapara la mayoría de productos del _dataset_. No obstante, se considera que cada grupo contiene una cantidad de datos suficiente para poder entrenar un modelo de aprendizaje automático robusto. Por otro lado, para recapitular se listan los _clusters_ finales y cómo se han formado.

<ul>
    <li><i>alta_rotacion</i> &#8594; Representa el <i>cluster</i> 4 original</li>
    <li><i>estandar</i> &#8594; Agrupa los grupo 1, 5 y 7 originales</li>
    <li><i>residual</i> &#8594; Recoge los grupos 2, 3, 8 y 9 originales</li>
    <li><i>top_ventas</i> &#8594; Fusión de los <i>clusters</i> 0 y 6</li>
</ul>

In [ ]:
# Se añaden las etiquetas de los clusters al dataset reconstruido
añadir: pd.DataFrame = subset[["producto", "cluster_final", "cluster_nombre"]]
dataset_final: pd.DataFrame = dataset_reconstruido.merge(añadir, on="producto", how="inner")

In [ ]:
# Se dicotomizan las categorías de las variables mes_int y dia_semana_str y se exporta el dataset
dataset_final["mes_str"] = dataset_final["mes_int"].astype(str)
dataset_final: pd.DataFrame = pd.get_dummies(dataset_final, columns=["mes_str", "dia_semana_str"])

dataset_final.to_csv("dataset_preprocesado.csv")